In [1]:
import os

# hide broken GPU0, expose only physical GPU1
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
os.environ["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"

# optional, avoids NVML-based check weirdness in some setups
os.environ["PYTORCH_NVML_BASED_CUDA_CHECK"] = "0"

In [2]:
import torch
print("cuda?", torch.cuda.is_available(), "n_dev", torch.cuda.device_count())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

cuda? True n_dev 1
NVIDIA GeForce RTX 4090


In [3]:
import os, sys, math, time, random
import numpy as np
import pandas as pd

SEED = 0
def seed_everything(seed=0):
    random.seed(seed)
    np.random.seed(seed)
    try:
        import torch
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything(SEED)

# Paths (match your existing notebook default)
DG_H5AD = "read_data/datasets/dentate_gyrus/DentateGyrus.h5ad"

# Your provided code files (put them next to this notebook, or set absolute paths)
COUNT_FM_PY = "count_FM_function_v3.py"
MODELS_PY   = "models.py"

# Train/test split
TEST_FRAC = 0.2

# Comparison budget
N_GEN = None  # if None, generate same number as X_test
DEVICE = "cuda"  # "cuda" or "cpu"

# Featurizer for metrics (shared across models)
TARGET_SUM = 1e4
G_KEEP = 256   # select top-variance genes before PCA
PCA_DIM = 50   # set None to disable PCA (not recommended)

print("Config loaded.")

Config loaded.


# 1. Load Data

In [4]:
import scanpy as sc
import scipy.sparse as sp

adata = sc.read_h5ad(DG_H5AD)

# Prefer your layer if present
if "matrix" in adata.layers:
    X = adata.layers["matrix"]
else:
    X = adata.X

if sp.issparse(X):
    X = X.toarray()

# Ensure integer-like counts
X = np.asarray(X)
if not np.issubdtype(X.dtype, np.integer):
    # some h5ad store counts as float
    X = np.rint(X).astype(np.int64)

N, d = X.shape
print("Loaded X:", X.shape, X.dtype, "min/max:", X.min(), X.max())

Loaded X: (2930, 13913) int64 min/max: 0 1718


## train-test split

In [5]:
idx = np.arange(N)
rng = np.random.default_rng(SEED)
rng.shuffle(idx)

n_test = int(round(TEST_FRAC * N))
test_idx = idx[:n_test]
train_idx = idx[n_test:]

X_train = X[train_idx]
X_test  = X[test_idx]

print("Train:", X_train.shape, "Test:", X_test.shape)

Train: (2344, 13913) Test: (586, 13913)


## featurizer

In [6]:
from sklearn.decomposition import PCA

def log1p_libnorm_np(X_counts, target_sum=1e4):
    X_counts = X_counts.astype(np.float32, copy=False)
    lib = X_counts.sum(axis=1, keepdims=True)
    lib = np.maximum(lib, 1.0).astype(np.float32)
    Xn = X_counts * (float(target_sum) / lib)
    return np.log1p(Xn)

class Featurizer:
    def __init__(self, target_sum=1e4, g_keep=256, pca_dim=50, seed=0):
        self.target_sum = target_sum
        self.g_keep = g_keep
        self.pca_dim = pca_dim
        self.seed = seed
        self.top_idx = None
        self.pca = None

    def fit(self, X_real_counts):
        X_log = log1p_libnorm_np(X_real_counts, self.target_sum)
        gene_var = X_log.var(axis=0)
        g_keep = min(self.g_keep, X_log.shape[1])
        self.top_idx = np.argsort(gene_var)[-g_keep:]

        Z = X_log[:, self.top_idx]
        if self.pca_dim is None:
            self.pca = None
            return self

        p = min(self.pca_dim, Z.shape[1], Z.shape[0] - 1)
        self.pca = PCA(n_components=p, svd_solver="randomized", random_state=self.seed)
        self.pca.fit(Z)
        return self

    def transform(self, X_counts):
        X_log = log1p_libnorm_np(X_counts, self.target_sum)
        Z = X_log[:, self.top_idx]
        if self.pca is None:
            return Z
        return self.pca.transform(Z)

feat = Featurizer(target_sum=TARGET_SUM, g_keep=G_KEEP, pca_dim=PCA_DIM, seed=SEED).fit(X_train)
Z_test = feat.transform(X_test)
print("Feature dim:", Z_test.shape)

Feature dim: (586, 50)


# 2. metrics

In [7]:
import torch

def _cov_np(Z):
    Z = np.asarray(Z, dtype=np.float64)
    mu = Z.mean(axis=0)
    Xc = Z - mu[None, :]
    cov = (Xc.T @ Xc) / max(Z.shape[0] - 1, 1)
    return mu, cov

def _sqrtm_psd(A, eps=1e-12):
    w, V = np.linalg.eigh(A)
    w = np.clip(w, 0.0, None)
    return (V * np.sqrt(w + eps)) @ V.T

def w2_gaussian(Z_real, Z_fake):
    mu1, C1 = _cov_np(Z_real)
    mu2, C2 = _cov_np(Z_fake)

    diff = mu1 - mu2
    diff2 = float(diff @ diff)

    sqrtC1 = _sqrtm_psd(C1)
    A = sqrtC1 @ C2 @ sqrtC1
    sqrtA = _sqrtm_psd(A)

    tr = np.trace(C1 + C2 - 2.0 * sqrtA)
    w2_sq = diff2 + float(tr)
    w2_sq = max(w2_sq, 0.0)
    return float(np.sqrt(w2_sq))

def _median_heuristic_sigma2(Z, max_points=1000, seed=0):
    rng = np.random.default_rng(seed)
    n = Z.shape[0]
    m = min(n, max_points)
    idx = rng.choice(n, size=m, replace=False)
    X = torch.tensor(Z[idx], dtype=torch.float32)
    x2 = (X**2).sum(dim=1, keepdim=True)
    dist2 = x2 + x2.T - 2 * (X @ X.T)
    dist2 = dist2.flatten()
    dist2 = dist2[dist2 > 0]
    return float(torch.median(dist2).item())

def mmd2_rbf_unbiased(Z_real, Z_fake, sigma2=None, device=None):
    if device is None:
        device = "cuda" if torch.cuda.is_available() else "cpu"

    X = torch.tensor(Z_real, dtype=torch.float32, device=device)
    Y = torch.tensor(Z_fake, dtype=torch.float32, device=device)

    m = X.shape[0]
    n = Y.shape[0]

    if sigma2 is None:
        sigma2 = _median_heuristic_sigma2(np.vstack([Z_real, Z_fake]), seed=SEED)
        sigma2 = max(sigma2, 1e-6)

    def k_rbf(A, B):
        a2 = (A**2).sum(dim=1, keepdim=True)
        b2 = (B**2).sum(dim=1, keepdim=True).T
        dist2 = a2 + b2 - 2.0 * (A @ B.T)
        return torch.exp(-dist2 / (2.0 * sigma2))

    Kxx = k_rbf(X, X)
    Kyy = k_rbf(Y, Y)
    Kxy = k_rbf(X, Y)

    Kxx.fill_diagonal_(0.0)
    Kyy.fill_diagonal_(0.0)

    term_x = Kxx.sum() / (m * (m - 1))
    term_y = Kyy.sum() / (n * (n - 1))
    term_xy = Kxy.mean()

    mmd2 = term_x + term_y - 2.0 * term_xy
    return float(mmd2.detach().cpu().item()), float(sigma2)

# 3. Models

In [8]:
class UncondModel:
    name = "base"
    def fit(self, X_train_counts, **kwargs):
        raise NotImplementedError
    def sample(self, n_samples, **kwargs):
        raise NotImplementedError

## 3.1 count-FM

In [9]:
import importlib.util
import types

def import_from_path(py_path, module_name):
    spec = importlib.util.spec_from_file_location(module_name, py_path)
    mod = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = mod
    spec.loader.exec_module(mod)
    return mod

import ot

countfm = import_from_path(COUNT_FM_PY, "countfm_user")
models  = import_from_path(MODELS_PY,   "models_user")

# Make tqdm render as a notebook widget (updates in-place, no spam lines)
from tqdm.notebook import tqdm as tqdm_nb
countfm.tqdm = tqdm_nb

import torch

torch.set_default_device("cpu")

class CountFMModel(UncondModel):
    name = "count-FM"

    def __init__(
        self,
        device="cuda",
        num_epochs=500,
        batch_size=32,
        eps_t=1e-4,
        eps_log=1e-8,
        loss_mode="poisson",
        margin=2,
        C_max=None,
        # match 3_scRNA.ipynb behavior
        x0_mode_train="poisson",
        lam0_train=1.0,
        # OT coupling option
        ot_cost="none",
        # transformer backbone
        d_model=256,
        depth=8,
        n_heads=8,
        mlp_ratio=4.0,
        dropout=0.0,
        chunk_size=128,
        log_cap=50.0,
    ):
        self.device = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")

        self.num_epochs = num_epochs
        self.batch_size = batch_size
        self.eps_t = eps_t
        self.eps_log = eps_log
        self.loss_mode = loss_mode
        self.margin = margin
        self.C_max = C_max

        self.x0_mode_train = x0_mode_train
        self.lam0_train = lam0_train
        self.ot_cost = ot_cost

        self.d_model = d_model
        self.depth = depth
        self.n_heads = n_heads
        self.mlp_ratio = mlp_ratio
        self.dropout = dropout
        self.chunk_size = chunk_size
        self.log_cap = log_cap

        self.net = None
        self.d = None
        self.train_loss = None

        self.lr = 1e-4
        self.weight_decay = 0.0

    def fit(self, X_train_counts, lr=None, weight_decay=None, num_epochs=None, resume=True):
        lr = float(self.lr if lr is None else lr)
        weight_decay = float(self.weight_decay if weight_decay is None else weight_decay)
        num_epochs = int(self.num_epochs if num_epochs is None else num_epochs)

        X1_torch = torch.tensor(X_train_counts, dtype=torch.float32)  # CPU
        d = X1_torch.shape[1]
        self.d = d

        if (not resume) or (self.net is None):
            net_base = models.ChunkedAdaLNTransformer(
                dim=d, out_dim=2 * d, time_varying=True,
                d_model=self.d_model, depth=self.depth, n_heads=self.n_heads,
                mlp_ratio=self.mlp_ratio, dropout=self.dropout, chunk_size=self.chunk_size,
            ).to(self.device)

            self.net = models.ChunkedAdaLNTransformer_rate(
                net_base, log_cap=self.log_cap, eps=self.eps_log
            ).to(self.device)

        opt = torch.optim.AdamW(self.net.parameters(), lr=lr, weight_decay=weight_decay)

        out = countfm.CountFM_train(
            X1_torch=X1_torch,
            nets=self.net,
            optimizer=opt,
            num_epochs=num_epochs,
            batch_size=self.batch_size,
            device=self.device,
            separate_heads=False,
            X0_torch=None,
            x0_mode=self.x0_mode_train,
            ot_cost=self.ot_cost,
            C_max=self.C_max,
            margin=self.margin,
            eps_t=self.eps_t,
            eps_log=self.eps_log,
            loss_mode=self.loss_mode,
        )

        if isinstance(out, tuple):
            self.net, self.train_loss = out
        else:
            self.net, self.train_loss = out, None

        print(f"count-FM: resume={resume} lr={lr} wd={weight_decay} epochs={num_epochs}")
        return self

    @torch.no_grad()
    def sample(self, n_samples, n_step=500, batch_eval=32, x0_mode="poisson", lam0=1.0):
        import torch
        assert self.net is not None
        assert self.d is not None
        d = int(self.d)

        if x0_mode == "poisson":
            x0 = torch.poisson(torch.full((n_samples, d), float(lam0), dtype=torch.float32)).to(torch.int64)
        elif x0_mode == "uniform":
            # only if you really want it
            C_max_eff = int(lam0)  # dummy fallback
            x0 = torch.randint(0, C_max_eff + 1, (n_samples, d), dtype=torch.int64)
        else:
            raise ValueError(f"Unknown x0_mode: {x0_mode}")

        x1, _ = sample_euler_batched(
            net=self.net,
            n_step=n_step,
            x0_cpu=x0,
            device=self.device,
            batch_eval=batch_eval,
            eps_t=self.eps_t,
            eps_log=self.eps_log,
        )
        return x1.numpy().astype(np.int64)

@torch.no_grad()
def sample_euler_batched(net, n_step, x0_cpu, device, batch_eval=32, eps_t=1e-4, eps_log=1e-8):
    net.eval()
    N, d = x0_cpu.shape
    outs = []
    Delta = (1.0 - 2.0 * eps_t) / float(n_step)
    Delta_t = torch.tensor(Delta, device=device, dtype=torch.float32)

    for start in range(0, N, batch_eval):
        end = min(start + batch_eval, N)
        xt = x0_cpu[start:end].to(device=device, dtype=torch.float32).clone()
        B = xt.shape[0]
        t = torch.full((B, 1), eps_t, device=device, dtype=torch.float32)

        for _ in range(n_step):
            xt_t = torch.cat([xt, t], dim=1)
            out = net(xt_t)
            lambda_theta, beta_theta = out[:, :d], out[:, d:]

            idx_0 = (xt <= 0).to(torch.float32)
            mu_theta = (xt * beta_theta) * (1.0 - idx_0)

            r_i = lambda_theta + mu_theta
            p_none  = torch.exp(-r_i * Delta_t)
            p_jump  = 1.0 - p_none
            p_birth = p_jump * (lambda_theta / (r_i + eps_log))
            p_death = p_jump * (mu_theta     / (r_i + eps_log))

            probs3 = torch.stack([p_none, p_birth, p_death], dim=-1)
            choice = torch.multinomial(probs3.reshape(-1, 3), 1).view(B, d)
            adj = (choice == 1).to(torch.float32) - (choice == 2).to(torch.float32)
            xt = torch.clamp(xt + adj, min=0.0)

            t = torch.minimum(t + Delta_t, torch.full_like(t, 1.0 - eps_t))

        outs.append(xt.to(torch.long).cpu())

        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    return torch.cat(outs, dim=0), None

## 3.2 scVI

In [10]:
import torch
import anndata as ad
import scvi

class SCVIModel(UncondModel):
    name = "scVI"
    def __init__(self, device="cuda", n_latent=10, n_layers=2, n_hidden=128,
                 gene_likelihood="zinb", max_epochs=400):
        self.device = device
        self.n_latent = n_latent
        self.n_layers = n_layers
        self.n_hidden = n_hidden
        self.gene_likelihood = gene_likelihood
        self.max_epochs = max_epochs
        self.model = None

        self.lr = 1e-3
        self.weight_decay = 1e-6

    def fit(self, X_train_counts, lr=None, weight_decay=None, max_epochs=None, resume=True):
        lr = float(self.lr if lr is None else lr)
        weight_decay = float(self.weight_decay if weight_decay is None else weight_decay)
        max_epochs = int(self.max_epochs if max_epochs is None else max_epochs)

        if (not resume) or (self.model is None):
            adata_train = ad.AnnData(X_train_counts.astype(np.int64))
            scvi.model.SCVI.setup_anndata(adata_train)
            self.model = scvi.model.SCVI(
                adata_train,
                n_latent=self.n_latent,
                n_layers=self.n_layers,
                n_hidden=self.n_hidden,
                gene_likelihood=self.gene_likelihood,
                use_observed_lib_size=False,
            )

        try:
            self.model.train(
                max_epochs=max_epochs,
                accelerator="gpu" if (self.device == "cuda" and torch.cuda.is_available()) else "cpu",
                devices=1,
                plan_kwargs={"lr": lr, "weight_decay": weight_decay},
            )
        except TypeError:
            self.model.train(
                max_epochs=max_epochs,
                accelerator="gpu" if (self.device == "cuda" and torch.cuda.is_available()) else "cpu",
                devices=1,
                lr=lr,
                weight_decay=weight_decay,
            )

        print(f"scVI: resume={resume} lr={lr} wd={weight_decay} epochs={max_epochs}")
        return self

    @torch.no_grad()
    def sample(self, n_samples):
        assert self.model is not None
        m = self.model
        module = m.module
        dev = m.device
    
        # latent prior
        n_latent = getattr(module, "n_latent", None)
        if n_latent is None:
            n_latent = self.n_latent
        z = torch.randn((n_samples, int(n_latent)), device=dev)
    
        # batch index (Dentate Gyrus is single batch in your setup)
        batch_index = torch.zeros((n_samples, 1), dtype=torch.long, device=dev)
    
        # library prior (works when use_observed_lib_size=False)
        lib_means = getattr(module, "library_log_means", None)
        lib_vars  = getattr(module, "library_log_vars", None)
    
        if lib_means is not None and lib_vars is not None:
            lib_means = torch.as_tensor(lib_means, dtype=torch.float32, device=dev).view(-1)
            lib_vars  = torch.as_tensor(lib_vars,  dtype=torch.float32, device=dev).view(-1)
    
            # pick batch-specific params (here always 0)
            b = batch_index.view(-1)
            mu  = lib_means[b].view(-1, 1)
            var = lib_vars[b].view(-1, 1)
            library = torch.randn((n_samples, 1), device=dev) * torch.sqrt(var + 1e-8) + mu
        else:
            # fallback: empirical log library from training data
            Xtr = m.adata.X
            if hasattr(Xtr, "toarray"):
                Xtr = Xtr.toarray()
            log_lib = np.log(np.maximum(np.asarray(Xtr).sum(axis=1), 1.0))
            mu = float(np.mean(log_lib))
            var = float(np.var(log_lib))
            library = torch.randn((n_samples, 1), device=dev) * math.sqrt(var + 1e-8) + mu
    
        # decode and sample counts
        outs = module.generative(z=z, library=library, batch_index=batch_index)
        px = outs["px"]
        x = px.sample()
    
        x = torch.clamp(x, min=0.0)
        x = x.detach().cpu().numpy()
        return np.rint(x).astype(np.int64)

## 3.3 scDiffusion

In [11]:
# !pip -q install blobfile mpi4py
# !git clone https://github.com/EperLuo/scDiffusion scDiffusion

In [12]:
import os, sys, glob, re, subprocess, time
import numpy as np
import scipy.sparse as sp
import anndata as ad
import torch

class ScDiffusionModel(UncondModel):
    name = "scDiffusion"

    def __init__(
        self,
        scd_root="scDiffusion",
        workdir="saved_models/scdiffusion_dg",
        DG_H5AD=None,                  # optional reference h5ad for var_names
        scimilarity_dir=None,
        device="cuda",
        seed=0,
        ae_steps=150000,
        diff_steps=600000,
        batch_size=128,
        ae_ckpt_freq=50000,
        diff_save_interval=200000,
        lr=1e-4,
        weight_decay=1e-4,
        latent_dim=128,
        hidden_dim=(512, 512, 256, 128),
        noise_schedule="linear",
        diffusion_steps=1000,
    ):
        self.scd_root = scd_root
        self.workdir = workdir
        self.DG_H5AD = DG_H5AD
        self.scimilarity_dir = scimilarity_dir
        self.device = device
        self.seed = int(seed)

        self.ae_steps = int(ae_steps)
        self.diff_steps = int(diff_steps)
        self.batch_size = int(batch_size)
        self.ae_ckpt_freq = int(ae_ckpt_freq)
        self.diff_save_interval = int(diff_save_interval)

        self.lr = float(lr)
        self.weight_decay = float(weight_decay)

        self.latent_dim = int(latent_dim)
        self.hidden_dim = list(hidden_dim)
        self.noise_schedule = str(noise_schedule)
        self.diffusion_steps = int(diffusion_steps)

        # full gene dim (your notebook X_train dim)
        self.d_full = None
        # filtered gene dim after scDiffusion filter_genes/filter_cells
        self.d = None

        self.keep_gene_idx = None
        self.keep_cell_idx = None

        self.data_h5ad = None
        self.ae_ckpt = None
        self.backbone_ckpt = None
        self._train_lib_sizes = None

    def _ensure_repo_on_path(self):
        root = os.path.abspath(self.scd_root)
        if root not in sys.path:
            sys.path.insert(0, root)

    @staticmethod
    def _latest_step(paths, pat=r"step=(\d+)\.pt"):
        best = (-1, None)
        for p in paths:
            m = re.search(pat, os.path.basename(p))
            if m:
                s = int(m.group(1))
                if s > best[0]:
                    best = (s, p)
        return best  # (step, path)

    @staticmethod
    def _latest_model_step(paths, pat=r"model(\d+)\.pt"):
        best = (-1, None)
        for p in paths:
            m = re.search(pat, os.path.basename(p))
            if m:
                s = int(m.group(1))
                if s > best[0]:
                    best = (s, p)
        return best  # (step, path)

    def _write_train_h5ad(self, X_train_counts, h5ad_path):
        os.makedirs(os.path.dirname(h5ad_path), exist_ok=True)

        Xc = np.asarray(X_train_counts)
        Xc = np.rint(Xc).astype(np.int64, copy=False)
        Xsp = sp.csr_matrix(Xc)
        adata_local = ad.AnnData(Xsp)

        # optional: preserve gene names (helps reproducibility)
        if self.DG_H5AD is not None:
            import scanpy as sc
            ad0 = sc.read_h5ad(self.DG_H5AD)
            ad0.var_names_make_unique()
            if ad0.shape[1] == adata_local.shape[1]:
                adata_local.var_names = ad0.var_names

        adata_local.obs["celltype"] = "all"
        adata_local.write_h5ad(h5ad_path)
        return h5ad_path

    def _infer_scdiff_filter_indices(self):
        import scanpy as sc

        ad0 = sc.read_h5ad(self.data_h5ad)
        ad0.var_names_make_unique()

        orig_var = ad0.var_names.copy()
        orig_obs = ad0.obs_names.copy()

        # match scDiffusion loader behavior
        sc.pp.filter_genes(ad0, min_cells=3)
        sc.pp.filter_cells(ad0, min_genes=10)
        ad0.var_names_make_unique()

        keep_var = ad0.var_names
        keep_obs = ad0.obs_names

        keep_gene_idx = orig_var.get_indexer(keep_var)
        keep_cell_idx = orig_obs.get_indexer(keep_obs)

        if np.any(keep_gene_idx < 0) or np.any(keep_cell_idx < 0):
            raise RuntimeError("Failed to map filtered genes/cells back to original indices.")

        self.keep_gene_idx = keep_gene_idx.astype(int)
        self.keep_cell_idx = keep_cell_idx.astype(int)
        self.d = int(len(self.keep_gene_idx))

        print("[scDiff] d_full =", self.d_full,
              "d_after_filter =", self.d,
              "n_train_full =", len(orig_obs),
              "n_after_filter =", len(self.keep_cell_idx))

    def _train_or_resume_ae(self, ae_dir, resume=True):
        self._ensure_repo_on_path()
        from guided_diffusion.cell_datasets_loader import load_data
        from VAE.VAE_model import VAE

        dev = torch.device("cuda" if (self.device == "cuda" and torch.cuda.is_available()) else "cpu")
        torch.manual_seed(self.seed)
        np.random.seed(self.seed)

        os.makedirs(ae_dir, exist_ok=True)

        ae_pat = os.path.join(ae_dir, f"model_seed={self.seed}_step=*.pt")
        ae_cands = glob.glob(ae_pat)
        last_step, last_path = self._latest_step(ae_cands, pat=r"step=(\d+)\.pt")

        target_step = self.ae_steps - 1
        if resume and last_step >= target_step:
            self.ae_ckpt = last_path
            return

        # IMPORTANT: num_genes must match filtered gene count (self.d)
        vae = VAE(
            num_genes=int(self.d),
            device=str(dev),
            seed=self.seed,
            loss_ae="mse",
            hidden_dim=self.latent_dim,
            decoder_activation="ReLU",
        )

        # force fp32 batches (loader can yield float64)
        _orig_train_step = vae.train_step
        def _train_step_fp32(genes):
            if isinstance(genes, torch.Tensor) and genes.dtype != torch.float32:
                genes = genes.to(dtype=torch.float32)
            return _orig_train_step(genes)
        vae.train_step = _train_step_fp32

        if resume and last_step >= 0:
            vae.load_state_dict(torch.load(last_path, map_location="cpu"))
            start = last_step + 1
        else:
            if self.scimilarity_dir is not None:
                use_gpu = (dev.type == "cuda")
                vae.encoder.load_state(os.path.join(self.scimilarity_dir, "encoder.ckpt"), use_gpu=use_gpu)
                vae.decoder.load_state(os.path.join(self.scimilarity_dir, "decoder.ckpt"), use_gpu=use_gpu)
            else:
                print("[AE] training from scratch (no scimilarity init)")
            start = 0

        data = load_data(data_dir=self.data_h5ad, batch_size=self.batch_size, train_vae=True)
        freq = max(1, min(self.ae_ckpt_freq, target_step))

        t0 = time.time()
        for step in range(start, target_step + 1):
            genes, _ = next(data)
            stats = vae.train_step(genes)

            if step == start:
                try:
                    print("[AE] first batch shape:", tuple(genes.shape), "dtype:", genes.dtype)
                except Exception:
                    pass

            if step % 1000 == 0:
                print("[AE] step", step, "loss", stats["loss_reconstruction"], "elapsed", round(time.time()-t0, 1), "sec")

            if (step % freq == 0) or (step == target_step):
                torch.save(vae.state_dict(), os.path.join(ae_dir, f"model_seed={self.seed}_step={step}.pt"))

        self.ae_ckpt = os.path.join(ae_dir, f"model_seed={self.seed}_step={target_step}.pt")

    def fit(self, X_train_counts, lr=None, weight_decay=None, resume=True):
        self._ensure_repo_on_path()
        lr = float(self.lr if lr is None else lr)
        weight_decay = float(self.weight_decay if weight_decay is None else weight_decay)

        Xc = np.asarray(X_train_counts)
        self.d_full = int(Xc.shape[1])

        os.makedirs(self.workdir, exist_ok=True)
        self.data_h5ad = self._write_train_h5ad(
            X_train_counts,
            os.path.join(self.workdir, "train_for_scdiffusion.h5ad"),
        )

        self._infer_scdiff_filter_indices()

        # library sizes for sampling computed on kept cells + kept genes
        X_kept = Xc[self.keep_cell_idx][:, self.keep_gene_idx]
        self._train_lib_sizes = np.rint(X_kept.sum(axis=1)).astype(np.int64)

        ae_dir = os.path.join(self.workdir, "AE")
        backbone_dir = os.path.join(self.workdir, "backbone")
        model_name = "dg_scdiffusion"
        out_dir = os.path.join(backbone_dir, model_name)
        os.makedirs(out_dir, exist_ok=True)

        self._train_or_resume_ae(ae_dir, resume=resume)

        target_model = os.path.join(out_dir, f"model{self.diff_steps:06d}.pt")
        if resume and os.path.exists(target_model):
            self.backbone_ckpt = target_model
            return self

        resume_ckpt = ""
        if resume:
            cand = glob.glob(os.path.join(out_dir, "model*.pt"))
            last_step, last_path = self._latest_model_step(cand, pat=r"model(\d+)\.pt")
            if last_step >= 0 and last_step < self.diff_steps:
                resume_ckpt = last_path

        cmd = [
            sys.executable,
            os.path.join(self.scd_root, "cell_train.py"),
            "--data_dir", self.data_h5ad,
            "--vae_path", self.ae_ckpt,
            "--model_name", model_name,
            "--save_dir", backbone_dir,
            "--batch_size", str(self.batch_size),
            "--lr", str(lr),
            "--weight_decay", str(weight_decay),
            "--lr_anneal_steps", str(self.diff_steps),
            "--save_interval", str(self.diff_save_interval),
            "--input_dim", str(self.latent_dim),
            "--noise_schedule", str(self.noise_schedule),
            "--diffusion_steps", str(self.diffusion_steps),
        ]
        if resume_ckpt:
            cmd += ["--resume_checkpoint", resume_ckpt]

        print("[Diffusion] running:")
        print(" ".join(cmd))
        
        # subprocess.run(cmd, check=True)
        log_path = os.path.join(self.workdir, "diffusion_train.log")
        with open(log_path, "w") as f:
            p = subprocess.Popen(cmd, stdout=f, stderr=subprocess.STDOUT, text=True)
            print("Diffusion running, PID =", p.pid, "log =", log_path)
            ret = p.wait()
            if ret != 0:
                raise RuntimeError(f"scDiffusion diffusion stage failed, see {log_path}")
        

        if os.path.exists(target_model):
            self.backbone_ckpt = target_model
        else:
            cand = glob.glob(os.path.join(out_dir, "model*.pt"))
            _, last_path = self._latest_model_step(cand, pat=r"model(\d+)\.pt")
            if last_path is None:
                raise FileNotFoundError("No diffusion checkpoint found in " + out_dir)
            self.backbone_ckpt = last_path

        return self

    @torch.no_grad()
    def sample(self, n_samples, batch_size=1000):
        self._ensure_repo_on_path()
        if self.ae_ckpt is None or self.backbone_ckpt is None:
            raise ValueError("Run fit() first, or load bundle first.")
        if self._train_lib_sizes is None or self.keep_gene_idx is None or self.d_full is None or self.d is None:
            raise ValueError("Missing training stats (need fit() or load bundle).")

        dev = torch.device("cuda" if (self.device == "cuda" and torch.cuda.is_available()) else "cpu")

        from VAE.VAE_model import VAE
        vae = VAE(
            num_genes=int(self.d),
            device=str(dev),
            seed=self.seed,
            loss_ae="mse",
            hidden_dim=self.latent_dim,
            decoder_activation="ReLU",
        )
        vae.load_state_dict(torch.load(self.ae_ckpt, map_location="cpu"))
        vae.to(dev).eval()

        from guided_diffusion.script_util import create_model_and_diffusion, model_and_diffusion_defaults
        defaults = model_and_diffusion_defaults()
        defaults.update(dict(
            input_dim=self.latent_dim,
            hidden_dim=self.hidden_dim,
            diffusion_steps=self.diffusion_steps,
            noise_schedule=self.noise_schedule,
            class_cond=False,
        ))
        model, diffusion = create_model_and_diffusion(**defaults)
        model.load_state_dict(torch.load(self.backbone_ckpt, map_location="cpu"))
        model.to(dev).eval()

        n = int(n_samples)
        bs = int(batch_size)

        lat_list, got = [], 0
        while got < n:
            cur = min(bs, n - got)
            
            lat = diffusion.p_sample_loop(model,
                                          (cur, self.latent_dim), 
                                          clip_denoised=False, model_kwargs={})
            
            if isinstance(lat, tuple):
                lat = lat[0]
            elif isinstance(lat, dict) and "sample" in lat:
                lat = lat["sample"]
            
            lat_list.append(lat.detach().cpu().numpy())
            
            # lat = diffusion.p_sample_loop(model, (cur, self.latent_dim), clip_denoised=False, model_kwargs={})
            # lat_list.append(lat.detach().cpu().numpy())
            got += cur
        lat = np.concatenate(lat_list, axis=0)

        # decode to log1p-scale outputs, then expm1 back to relative expression
        x_log = vae(torch.tensor(lat, device=dev, dtype=torch.float32), return_decoded=True).detach().cpu().numpy()
        x_rel = np.expm1(np.maximum(x_log, 0.0)).astype(np.float64)
        probs = x_rel / np.maximum(x_rel.sum(axis=1, keepdims=True), 1e-12)

        rng = np.random.default_rng(self.seed)
        libs = self._train_lib_sizes
        libs = libs[libs > 0]
        if libs.size == 0:
            libs = np.array([1000], dtype=np.int64)

        Xf = np.empty((n, int(self.d)), dtype=np.int64)
        for i in range(n):
            L = int(rng.choice(libs))
            Xf[i] = rng.multinomial(L, probs[i])

        # pad back to full gene dimension
        Xfull = np.zeros((n, int(self.d_full)), dtype=np.int64)
        Xfull[:, self.keep_gene_idx] = Xf
        return Xfull

## 3.4 CFGen

In [13]:
# %%bash
# set -e

# # If CFGen exists and is a git repo, update it. Otherwise, remove and re-clone.
# if [ -d CFGen/.git ]; then
#   git -C CFGen fetch --all
#   git -C CFGen checkout main
#   git -C CFGen pull
# else
#   rm -rf CFGen
#   git clone https://github.com/theislab/CFGen CFGen
# fi

# # Patch packaging: setup.py expects README.rst but repo has README.md
# if [ -f CFGen/README.md ] && [ ! -f CFGen/README.rst ]; then
#   cp CFGen/README.md CFGen/README.rst
# fi

# pip install -e CFGen

# python -c "import cfgen; print('cfgen import OK:', cfgen.__file__)"

In [14]:
# !git clone https://github.com/theislab/CFGen CFGen
# !pip -q install -e CFGen

In [15]:
import os, json, math, shutil
import numpy as np
import pandas as pd
import torch

# pytorch_lightning import compatibility
try:
    import pytorch_lightning as pl
    from pytorch_lightning.callbacks import ModelCheckpoint
except Exception:
    import lightning.pytorch as pl
    from lightning.pytorch.callbacks import ModelCheckpoint

from anndata import AnnData

# CFGen imports
from cfgen.data.scrnaseq_loader import RNAseqLoader
from cfgen.models.base.encoder_model import EncoderModel
from cfgen.models.featurizers.category_featurizer import CategoricalFeaturizer
from cfgen.models.fm.denoising_model import MLPTimeStep
from cfgen.models.fm.fm import FM

def _torch_load_trusted(path, map_location="cpu"):
    # For your own checkpoints, allow full unpickling.
    # This avoids PyTorch 2.6 default weights_only=True failures.
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        # older torch that doesn't have weights_only
        return torch.load(path, map_location=map_location)


class CFGenModel(UncondModel):
    name = "CFGen"

    def __init__(
        self,
        device="cuda",
        workdir="saved_models/cfgen_dg_v1",
        seed=0,

        # We train "unconditional" by using a constant covariate with 1 category.
        covariate_key="dummy",

        # Dataset / loader
        layer_key="X_counts",
        normalization_type="log_gexp",
        subsample_frac=1.0,
        split_rates=(0.90, 0.10),

        # Encoder (AE) config (matches CFGen defaults)
        encoder_dims=(512, 256, 50),
        encoder_lr=1e-3,
        encoder_wd=1e-5,
        encoder_max_epochs=300,
        encoder_batch_size=256,

        # Flow (FM) config (matches CFGen defaults)
        fm_hidden_dim=32,
        fm_dropout_prob=0.0,
        fm_n_blocks=3,
        fm_embedding_dim=20,
        fm_conditional=False,          # IMPORTANT: unconditional training
        fm_normalization="none",
        fm_embed_size_factor=False,
        fm_conditioning_probability=0.8,
        fm_guided_conditioning=True,

        fm_lr=1e-4,
        fm_wd=1e-6,
        fm_sigma=1e-4,
        fm_use_ot=False,
        fm_antithetic_time_sampling=True,
        fm_max_epochs=1500,
        fm_batch_size=256,

        # Feature embedding
        one_hot_encode_features=False,
    ):
        self.device = device
        self.workdir = workdir
        self.seed = int(seed)

        self.covariate_key = str(covariate_key)
        self.layer_key = str(layer_key)
        self.normalization_type = str(normalization_type)
        self.subsample_frac = float(subsample_frac)
        self.split_rates = tuple(split_rates)

        self.encoder_dims = tuple(int(x) for x in encoder_dims)
        self.encoder_lr = float(encoder_lr)
        self.encoder_wd = float(encoder_wd)
        self.encoder_max_epochs = int(encoder_max_epochs)
        self.encoder_batch_size = int(encoder_batch_size)

        self.fm_hidden_dim = int(fm_hidden_dim)
        self.fm_dropout_prob = float(fm_dropout_prob)
        self.fm_n_blocks = int(fm_n_blocks)
        self.fm_embedding_dim = int(fm_embedding_dim)
        self.fm_conditional = bool(fm_conditional)
        self.fm_normalization = str(fm_normalization)
        self.fm_embed_size_factor = bool(fm_embed_size_factor)
        self.fm_conditioning_probability = float(fm_conditioning_probability)
        self.fm_guided_conditioning = bool(fm_guided_conditioning)

        self.fm_lr = float(fm_lr)
        self.fm_wd = float(fm_wd)
        self.fm_sigma = float(fm_sigma)
        self.fm_use_ot = bool(fm_use_ot)
        self.fm_antithetic_time_sampling = bool(fm_antithetic_time_sampling)
        self.fm_max_epochs = int(fm_max_epochs)
        self.fm_batch_size = int(fm_batch_size)

        self.one_hot_encode_features = bool(one_hot_encode_features)

        # trained artifacts
        self.encoder_model = None
        self.fm_model = None
        self._meta = None  # for save/load

        os.makedirs(self.workdir, exist_ok=True)

    def _dev(self):
        if self.device == "cuda" and torch.cuda.is_available():
            return torch.device("cuda")
        return torch.device("cpu")

    def _make_train_adata(self, X_train_counts, var_names=None):
        X = np.asarray(X_train_counts)
        if not np.issubdtype(X.dtype, np.integer):
            X = np.rint(X).astype(np.int64)

        adata = AnnData(X=X)
        if var_names is not None:
            adata.var_names = list(var_names)

        # constant covariate (1 category)
        adata.obs[self.covariate_key] = pd.Categorical(np.zeros((X.shape[0],), dtype=int))

        # CFGen loader expects a layer_key; if absent, it copies from X anyway, but keep explicit.
        adata.layers[self.layer_key] = adata.X.copy()
        return adata

    def fit(self, X_train_counts, var_names=None, resume=True):
        dev = self._dev()
        pl.seed_everything(self.seed, workers=True)

        # build AnnData in-memory (no muon read path needed)
        adata_tr = self._make_train_adata(X_train_counts, var_names=var_names)

        # dataset / loader
        ds = RNAseqLoader(
            adata_tr,
            layer_key=self.layer_key,
            covariate_keys=[self.covariate_key],
            subsample_frac=self.subsample_frac,
            normalization_type=self.normalization_type,
            is_binarized=False,
        )

        # split
        n = len(ds)
        n_tr = int(round(self.split_rates[0] * n))
        n_va = n - n_tr
        g = torch.Generator().manual_seed(self.seed)
        train_data, valid_data = torch.utils.data.random_split(ds, [n_tr, n_va], generator=g)

        dl_tr_enc = torch.utils.data.DataLoader(
            train_data, batch_size=self.encoder_batch_size, shuffle=True, num_workers=0, drop_last=True
        )
        dl_va_enc = torch.utils.data.DataLoader(
            valid_data, batch_size=self.encoder_batch_size, shuffle=False, num_workers=0, drop_last=True
        )

        # -------------------------
        # (1) Train encoder
        # -------------------------
        enc_ckpt_dir = os.path.join(self.workdir, "cfgen_encoder_ckpts")
        os.makedirs(enc_ckpt_dir, exist_ok=True)
        enc_last = os.path.join(enc_ckpt_dir, "last.ckpt")

        encoder_kwargs = {
            "rna": {
                "dims": list(self.encoder_dims),
                "batch_norm": True,
                "dropout": False,
                "dropout_p": 0.0,
            }
        }

        enc = EncoderModel(
            in_dim={"rna": ds.X["rna"].shape[1]},
            encoder_kwargs=encoder_kwargs,
            learning_rate=self.encoder_lr,
            weight_decay=self.encoder_wd,
            covariate_specific_theta=False,
            conditioning_covariate=self.covariate_key,
            n_cat=None,
            is_binarized=False,
        )

        enc_ckpt_cb = ModelCheckpoint(
            dirpath=enc_ckpt_dir,
            filename="epoch_{epoch:03d}",
            monitor="valid/loss",
            mode="min",
            every_n_epochs=20,
            save_last=True,
            auto_insert_metric_name=False,
        )

        enc_trainer = pl.Trainer(
            max_epochs=self.encoder_max_epochs,
            accelerator="gpu" if (dev.type == "cuda") else "cpu",
            devices=1,
            logger=False,
            callbacks=[enc_ckpt_cb],
            enable_checkpointing=True,
            log_every_n_steps=50,
        )

        ckpt_path = enc_last if (resume and os.path.exists(enc_last)) else None
        enc_trainer.fit(enc, train_dataloaders=dl_tr_enc, val_dataloaders=dl_va_enc, ckpt_path=ckpt_path)

        # freeze encoder for FM
        for p in enc.parameters():
            p.requires_grad = False
        enc.eval()

        # -------------------------
        # (2) Train FM in latent
        # -------------------------
        dl_tr_fm = torch.utils.data.DataLoader(
            train_data, batch_size=self.fm_batch_size, shuffle=True, num_workers=0, drop_last=True
        )
        dl_va_fm = torch.utils.data.DataLoader(
            valid_data, batch_size=self.fm_batch_size, shuffle=False, num_workers=0, drop_last=True
        )

        # categorical featurizer (still needed by CFGen code path)
        n_cat = len(ds.id2cov[self.covariate_key])
        feat = CategoricalFeaturizer(
            n_cat=n_cat,
            one_hot_encode_features=self.one_hot_encode_features,
            device=dev,
            embedding_dimensions=self.fm_embedding_dim,
        )

        feature_embeddings = {self.covariate_key: feat}

        denoise = MLPTimeStep(
            in_dim=self.encoder_dims[-1],
            hidden_dim=self.fm_hidden_dim,
            dropout_prob=self.fm_dropout_prob,
            n_blocks=self.fm_n_blocks,
            size_factor_min=ds.min_size_factor,
            size_factor_max=ds.max_size_factor,
            embed_size_factor=self.fm_embed_size_factor,
            covariate_list=[self.covariate_key],
            embedding_dim=self.fm_embedding_dim,
            normalization=self.fm_normalization,
            conditional=self.fm_conditional,
            is_binarized=False,
            modality_list=["rna"],
            conditioning_probability=self.fm_conditioning_probability,
            guided_conditioning=self.fm_guided_conditioning,
        ).to(dev)

        fm = FM(
            encoder_model=enc,
            denoising_model=denoise,
            feature_embeddings=feature_embeddings,
            plotting_folder=None,  # disable CFGen internal plotting
            in_dim={"rna": self.encoder_dims[-1]},
            size_factor_statistics={"mean": ds.log_size_factor_mu, "sd": ds.log_size_factor_sd},
            covariate_list=[self.covariate_key],
            theta_covariate=self.covariate_key,
            size_factor_covariate=self.covariate_key,
            encoder_type="fixed",
            learning_rate=self.fm_lr,
            weight_decay=self.fm_wd,
            antithetic_time_sampling=self.fm_antithetic_time_sampling,
            sigma=self.fm_sigma,
            covariate_specific_theta=False,
            plot_and_eval_every=10**9,  # effectively never
            use_ot=self.fm_use_ot,
            is_binarized=False,
            modality_list=["rna"],
            guidance_weights={self.covariate_key: 1},
        )

        fm_ckpt_dir = os.path.join(self.workdir, "cfgen_fm_ckpts")
        os.makedirs(fm_ckpt_dir, exist_ok=True)
        fm_last = os.path.join(fm_ckpt_dir, "last.ckpt")

        fm_ckpt_cb = ModelCheckpoint(
            dirpath=fm_ckpt_dir,
            filename="epoch_{epoch:03d}",
            monitor="valid/loss",
            mode="min",
            every_n_epochs=50,
            save_last=True,
            auto_insert_metric_name=False,
        )

        fm_trainer = pl.Trainer(
            max_epochs=self.fm_max_epochs,
            accelerator="gpu" if (dev.type == "cuda") else "cpu",
            devices=1,
            logger=False,
            callbacks=[fm_ckpt_cb],
            enable_checkpointing=True,
            log_every_n_steps=50,
            gradient_clip_val=1.0,
        )

        ckpt_path = fm_last if (resume and os.path.exists(fm_last)) else None
        fm_trainer.fit(fm, train_dataloaders=dl_tr_fm, val_dataloaders=dl_va_fm, ckpt_path=ckpt_path)

        fm.eval()

        # stash
        self.encoder_model = enc
        self.fm_model = fm

        self._meta = dict(
            d=int(ds.X["rna"].shape[1]),
            covariate_key=self.covariate_key,
            layer_key=self.layer_key,
            normalization_type=self.normalization_type,
            encoder_kwargs=encoder_kwargs,
            encoder_dims=self.encoder_dims,
            fm_params=dict(
                hidden_dim=self.fm_hidden_dim,
                dropout_prob=self.fm_dropout_prob,
                n_blocks=self.fm_n_blocks,
                embedding_dim=self.fm_embedding_dim,
                conditional=self.fm_conditional,
                normalization=self.fm_normalization,
                embed_size_factor=self.fm_embed_size_factor,
                conditioning_probability=self.fm_conditioning_probability,
                guided_conditioning=self.fm_guided_conditioning,
                lr=self.fm_lr,
                wd=self.fm_wd,
                sigma=self.fm_sigma,
                use_ot=self.fm_use_ot,
                antithetic_time_sampling=self.fm_antithetic_time_sampling,
                one_hot_encode_features=self.one_hot_encode_features,
            ),
            id2cov={self.covariate_key: list(ds.id2cov[self.covariate_key].keys())},
            size_factor_statistics=dict(mean=ds.log_size_factor_mu, sd=ds.log_size_factor_sd),
            min_size_factor=ds.min_size_factor,
            max_size_factor=ds.max_size_factor,
        )

        print(f"CFGen trained. workdir={self.workdir}  resume={resume}  device={dev}")
        return self

    @torch.no_grad()
    def sample(self, n_samples, n_steps=50, batch_size=512, unconditional=True):
        assert self.fm_model is not None, "Call fit() or load_cfgen_bundle() first."
        dev = self._dev()

        n = int(n_samples)
        bs = int(batch_size)
        reps = int(math.ceil(n / bs))

        # covariate_indices: constant 0 (since dummy covariate has 1 class)
        # covariate_indices = {self.covariate_key: torch.zeros((reps * bs,), dtype=torch.long)}
        covariate_indices = {self.covariate_key: torch.zeros((reps * bs,), dtype=torch.long, device=dev)}


        # --- make size_factor_statistics indexable (CFGen expects 1D tensors) ---
        sf = self.fm_model.size_factor_statistics
        
        def _as_1d_tensor(x):
            if torch.is_tensor(x):
                t = x.to(dev)
            else:
                t = torch.tensor(x, device=dev)
            if t.ndim == 0:
                t = t.reshape(1)
            return t
        
        for mod in self.fm_model.modality_list:
            for k in ("mean", "sd"):
                # sf[k] might be a float (bad), dict(mod->...), or dict(mod->dict(cov->...))
                if not isinstance(sf.get(k, None), dict):
                    sf[k] = {mod: {self.covariate_key: sf[k]}}
        
                if mod not in sf[k]:
                    # fallback if keys differ
                    sf[k][mod] = sf[k].get("rna", next(iter(sf[k].values())))
        
                if not isinstance(sf[k][mod], dict):
                    sf[k][mod] = {self.covariate_key: sf[k][mod]}
        
                if self.covariate_key not in sf[k][mod]:
                    sf[k][mod][self.covariate_key] = next(iter(sf[k][mod].values()))
        
                sf[k][mod][self.covariate_key] = _as_1d_tensor(sf[k][mod][self.covariate_key])
        


        
        out = self.fm_model.batched_sample(
            batch_size=bs,
            repetitions=reps,
            n_sample_steps=int(n_steps),
            theta_covariate=self.covariate_key,
            size_factor_covariate=self.covariate_key,
            conditioning_covariates=[self.covariate_key],
            covariate_indices=covariate_indices,
            log_size_factor=None,
            unconditional=bool(unconditional),
        )["rna"]

        X = out.cpu().numpy()[:n]
        X = np.asarray(X)
        if not np.issubdtype(X.dtype, np.integer):
            X = np.rint(X).astype(np.int64)
        return X

## 3.5 scLDM

In [16]:
import os, json, pickle, subprocess
import numpy as np
import pandas as pd
import anndata as ad

class SCLDMModel(UncondModel):
    name = "scLDM"

    def _maybe_reset_run(self, exp_name, reset_if_broken=True):
        ckpt_dir = os.path.join(self.workdir, "experiments", "checkpoints", exp_name)
        last_ckpt = os.path.join(ckpt_dir, "last.ckpt")
        cfg_yaml  = os.path.join(ckpt_dir, "config.yaml")
    
        # if folder exists but missing the required files, it's a broken run
        if os.path.isdir(ckpt_dir) and reset_if_broken:
            ok = os.path.exists(last_ckpt) and os.path.exists(cfg_yaml)
            if not ok:
                print(f"[scLDM] broken run detected, deleting {ckpt_dir}")
                import shutil
                shutil.rmtree(ckpt_dir, ignore_errors=True)
        

    def __init__(
        self,
        scldm_root="scldm",                 # local clone of https://github.com/czi-ai/scLDM
        py_exec=None,                       # IMPORTANT: python in a py3.11+ env with scldm installed
        workdir="saved_models/scldm_dg",
        seed=0,
        device="cuda",
        vae_epochs=100,
        ldm_epochs=100,
        batch_size=16,
        devices=1,
        test_batch_size=32,
        num_workers=0,
        timesteps=50,
        guidance_weight=0.0,                # we set a dummy label with scale 0.0 for unconditional
    ):
        self.scldm_root = scldm_root
        self.py_exec = py_exec if py_exec is not None else "python"
        # self.workdir = workdir
        self.workdir = os.path.abspath(workdir)
        self.seed = int(seed)
        # self.device = str(device)
        self.devices = int(devices)

        self.vae_epochs = int(vae_epochs)
        self.ldm_epochs = int(ldm_epochs)
        self.batch_size = int(batch_size)
        self.test_batch_size = int(test_batch_size)
        self.num_workers = int(num_workers)
        self.timesteps = int(timesteps)
        self.guidance_weight = float(guidance_weight)

        self.data_dir = os.path.join(self.workdir, "data")
        self.infer_dir = os.path.join(self.workdir, "inference")
        os.makedirs(self.data_dir, exist_ok=True)
        os.makedirs(self.infer_dir, exist_ok=True)

        # fixed dummy conditioning to satisfy scLDM code path, but keep it unconditional via guidance_weight=0.0
        # self.label_key = "dummy"
        # self.label_vocab = {self.label_key: 1}            # one category: "0"
        # self.guidance_dict = {self.label_key: self.guidance_weight}

        self.label_key = "clusters"
        self.label_vocab = {self.label_key: 1}            # one category: "0"
        self.guidance_dict = {self.label_key: self.guidance_weight}
        

        # filled after fit
        self.d = None
        self.var_names = None
        self.train_h5ad = None
        self.test_h5ad = None
        self.mu_pkl = None
        self.sd_pkl = None
        self.vae_name = None
        self.ldm_name = None
        self.ckpt_root = None                             # base_experiment_path/checkpoints
        self.vae_ckpt_dir = None
        self.ldm_ckpt_dir = None

    def _assert_repo(self):
        if not os.path.isdir(self.scldm_root):
            raise FileNotFoundError(
                f"scldm_root not found: {self.scldm_root}\n"
                f"Clone scLDM repo there, and set SCLDM_PY to a py3.11 env with scldm installed."
            )


    def _run(self, cmd, cwd):
        print("[scLDM] running:")
        print(" ".join(cmd))
    
        env = os.environ.copy()
        env["WANDB_MODE"] = "disabled"
        env["WANDB_DISABLED"] = "true"
        env["WANDB_SILENT"] = "true"
    
        # IMPORTANT: do NOT force CUDA_VISIBLE_DEVICES here (don’t set it to "0")
        # Let your global setting decide (you’re on GPU 1).
        # (So: no env["CUDA_VISIBLE_DEVICES"]=... in this function.)
    
        # Wrap python script execution to force torch.load(weights_only=False)
        if len(cmd) >= 2 and cmd[1].endswith(".py"):
            script = cmd[1]
            args = cmd[2:]
    
            # one-liner, no indentation
            pycode = (
                "import sys,runpy,torch;"
                "_real=torch.load;"
                "torch.load=lambda *a,**kw:_real(*a,**dict(kw,weights_only=False));"
                f"sys.argv=[r'{script}']+sys.argv[1:];"
                f"runpy.run_path(r'{script}',run_name='__main__')"
            )
    
            cmd2 = [cmd[0], "-c", pycode] + args
            subprocess.run(cmd2, cwd=cwd, check=True, env=env)
        else:
            subprocess.run(cmd, cwd=cwd, check=True, env=env)


    def _write_train_test_h5ad(self, X_train_counts, X_test_counts, var_names):
        Xtr = np.asarray(X_train_counts)
        Xte = np.asarray(X_test_counts)

        if not np.issubdtype(Xtr.dtype, np.integer):
            Xtr = np.rint(Xtr).astype(np.int64)
        if not np.issubdtype(Xte.dtype, np.integer):
            Xte = np.rint(Xte).astype(np.int64)

        self.d = int(Xtr.shape[1])
        self.var_names = list(var_names) if var_names is not None else [str(i) for i in range(self.d)]

        # scLDM expects categorical labels in .obs for each class key
        # dummy_tr = pd.Categorical(np.full((Xtr.shape[0],), "0"))
        # dummy_te = pd.Categorical(np.full((Xte.shape[0],), "0"))

        # ad_tr = ad.AnnData(X=Xtr)
        # ad_tr.var_names = self.var_names
        # ad_tr.obs[self.label_key] = dummy_tr

        # ad_te = ad.AnnData(X=Xte)
        # ad_te.var_names = self.var_names
        # ad_te.obs[self.label_key] = dummy_te

        # self.train_h5ad = os.path.join(self.data_dir, "train.h5ad")
        # self.test_h5ad  = os.path.join(self.data_dir, "test.h5ad")


        lab_tr = pd.Categorical(np.full((Xtr.shape[0],), "0"))
        lab_te = pd.Categorical(np.full((Xte.shape[0],), "0"))
        
        ad_tr = ad.AnnData(X=Xtr)
        ad_tr.var_names = self.var_names
        ad_tr.obs[self.label_key] = lab_tr
        
        ad_te = ad.AnnData(X=Xte)
        ad_te.var_names = self.var_names
        ad_te.obs[self.label_key] = lab_te
        
        self.train_h5ad = os.path.abspath(os.path.join(self.data_dir, "train.h5ad"))
        self.test_h5ad  = os.path.abspath(os.path.join(self.data_dir, "test.h5ad"))


        ad_tr.write(self.train_h5ad)
        ad_te.write(self.test_h5ad)

        # size factor stats per label class (required if you set mu_size_factor / sd_size_factor)
        lib = Xtr.sum(axis=1).astype(np.float64)
        lib = np.maximum(lib, 1.0)
        log_sf = np.log(lib)
        mu = float(log_sf.mean())
        sd = float(log_sf.std(ddof=1) if log_sf.size > 1 else 1.0)

        mu_dict = {self.label_key: {"0": mu}}
        sd_dict = {self.label_key: {"0": sd}}

        self.mu_pkl = os.path.join(self.data_dir, "log_size_factor_mu.pkl")
        self.sd_pkl = os.path.join(self.data_dir, "log_size_factor_sd.pkl")
        with open(self.mu_pkl, "wb") as f:
            pickle.dump(mu_dict, f)
        with open(self.sd_pkl, "wb") as f:
            pickle.dump(sd_dict, f)

    def _common_overrides(self):
        # We override dataset_params.dentate_gyrus so scLDM uses our gene set and our dummy label.
        # We also force genes_seq_len = d so generated X has full gene dimension for your evaluation.
        d = int(self.d)
        return [
            f"seed={self.seed}",
            "datamodule.dataset=dentate_gyrus",
            f"datamodule.datamodule.train_adata_path={self.train_h5ad}",
            f"datamodule.datamodule.test_adata_path={self.test_h5ad}",
            f"datamodule.vocabulary_encoder.adata_path={self.train_h5ad}",
            "datamodule.dataset_params.dentate_gyrus.metadata_json=null",
            f"datamodule.dataset_params.dentate_gyrus.n_genes={d}",
            f"datamodule.dataset_params.dentate_gyrus.genes_seq_len={d}",
            "datamodule.dataset_params.dentate_gyrus.sample_genes=all",
            f"datamodule.dataset_params.dentate_gyrus.class_vocab_sizes.{self.label_key}=1",
            f"datamodule.dataset_params.dentate_gyrus.guidance_weight.{self.label_key}={self.guidance_weight}",
            # f"datamodule.dataset_params.dentate_gyrus.class_vocab_sizes={{{self.label_key}:1}}",
            # f"datamodule.dataset_params.dentate_gyrus.guidance_weight={{{self.label_key}:{self.guidance_weight}}}",
            f"datamodule.dataset_params.dentate_gyrus.mu_size_factor={self.mu_pkl}",
            f"datamodule.dataset_params.dentate_gyrus.sd_size_factor={self.sd_pkl}",
            f"datamodule.datamodule.batch_size={self.batch_size}",
            f"datamodule.datamodule.test_batch_size={self.test_batch_size}",
            f"datamodule.datamodule.num_workers={self.num_workers}",
            f"model.batch_size={self.batch_size}",
            f"model.test_batch_size={self.test_batch_size}",
        ]

    def fit(self, X_train_counts, X_test_counts, var_names=None, resume=True):
        self._assert_repo()
        os.makedirs(self.workdir, exist_ok=True)

        self._write_train_test_h5ad(X_train_counts, X_test_counts, var_names=var_names)

        self.vae_name = "vae_dg_custom"
        self.ldm_name = "ldm_dg_custom"

        self._maybe_reset_run(self.vae_name, reset_if_broken=True)
        self._maybe_reset_run(self.ldm_name, reset_if_broken=True)

        # where scLDM will place checkpoints/configs
        base_experiment_path = os.path.abspath(os.path.join(self.workdir, "experiments"))
        self.ckpt_root = os.path.join(base_experiment_path, "checkpoints")
        self.vae_ckpt_dir = os.path.join(self.ckpt_root, self.vae_name)
        self.ldm_ckpt_dir = os.path.join(self.ckpt_root, self.ldm_name)

        # 1) VAE training (auto resumes from last.ckpt if present)
        cmd_vae = [
            self.py_exec,
            os.path.join("experiments", "scripts", "train.py"),
            f"paths.base_data_path={os.path.abspath(self.data_dir)}",
            f"paths.base_experiment_path={base_experiment_path}",
            f"experiment_name={self.vae_name}",
            f"training.num_epochs={self.vae_epochs}",
            "training.trainer.enable_progress_bar=true",
        ] + self._common_overrides()

        self._run(cmd_vae, cwd=self.scldm_root)

        # 2) LDM training (auto resumes from last.ckpt if present)
        cmd_ldm = [
            self.py_exec,
            os.path.join("experiments", "scripts", "train_ldm.py"),
            f"paths.base_data_path={os.path.abspath(self.data_dir)}",
            f"paths.base_experiment_path={base_experiment_path}",
            f"experiment_name={self.ldm_name}",
            f"training.num_epochs={self.ldm_epochs}",
            "training.trainer.enable_progress_bar=true",
            f"model.module.vae_as_tokenizer.load_from_checkpoint.ckpt_path={self.ckpt_root}",
            f"model.module.vae_as_tokenizer.load_from_checkpoint.job_name={self.vae_name}",
            "model.module.vae_as_tokenizer.load_from_checkpoint.epoch=null",
        ] + self._common_overrides()

        self._run(cmd_ldm, cwd=self.scldm_root)

        print(f"[scLDM] trained. workdir={self.workdir}  resume={resume}")
        return self

    def _dup_keys(self, names):
        counts = {}
        out = []
        for n in names:
            k = counts.get(n, 0)
            out.append(f"{n}__dup{k}")
            counts[n] = k + 1
        return np.asarray(out, dtype=str)

    def _canonical_var_names(self):
        if not hasattr(self, "_canon_var_names") or self._canon_var_names is None:
            if self.train_h5ad is None or (not os.path.exists(self.train_h5ad)):
                raise RuntimeError("[scLDM] train_h5ad missing, cannot infer canonical var_names.")
            ad0 = ad.read_h5ad(self.train_h5ad)
            self._canon_var_names = np.asarray(ad0.var_names, dtype=str)
        return self._canon_var_names
        

    def _infer_one_round(
        self,
        gen_idx=0,
        timesteps=None,
        test_batch_size=None,
        guidance_weight=None,
        unconditional=True,
    ):
        # inference.py saves: {inference_path}/{dataset}_generated_{idx}.h5ad
        ckpt_file = os.path.join(self.ldm_ckpt_dir, "last.ckpt")
        cfg_file  = os.path.join(self.ldm_ckpt_dir, "config.yaml")
        if not os.path.exists(ckpt_file) or not os.path.exists(cfg_file):
            raise FileNotFoundError("Missing scLDM LDM checkpoint/config. Run fit() first or load bundle.")
    
        tb = self.test_batch_size if (test_batch_size is None) else int(test_batch_size)
        ts = self.timesteps if (timesteps is None) else int(timesteps)
        gw = self.guidance_weight if (guidance_weight is None) else float(guidance_weight)

        cmd_inf = [
            self.py_exec,
            os.path.join("experiments", "scripts", "inference.py"),
            f"ckpt_file={ckpt_file}",
            f"config_file={cfg_file}",
            f"inference_path={os.path.abspath(self.infer_dir)}",
            f"dataset_generation_idx={int(gen_idx)}",
        
            # what you control at generation time
            f"datamodule.datamodule.test_batch_size={tb}",
            f"++model.module.generation_args.timesteps={ts}",
            f"++model.module.generation_args.guidance_weight={{{self.label_key}:{gw}}}",
        
            # keep this to avoid unrelated interpolation crash
            "++datamodule.dataset_params.homo_sapiens.metadata_genes=null",
        ] + self._common_overrides()

        

        # cmd_inf = [
        #     self.py_exec,
        #     os.path.join("experiments", "scripts", "inference.py"),
        #     f"ckpt_file={ckpt_file}",
        #     f"config_file={cfg_file}",
        #     f"inference_path={os.path.abspath(self.infer_dir)}",
        #     f"dataset_generation_idx={int(gen_idx)}",
        
        #     # REQUIRED for predict dataloader
        #     f"++datamodule.datamodule.train_adata_path={self.train_h5ad}",
        #     f"++datamodule.datamodule.test_adata_path={self.test_h5ad}",
        
        #     # make encoder read the same gene set / categories
        #     f"++datamodule.vocabulary_encoder.adata_path={self.train_h5ad}",
        
        #     # your local size-factor pkls (avoid /work/_artifacts)
        #     f"++datamodule.dataset_params.dentate_gyrus.mu_size_factor={self.mu_pkl}",
        #     f"++datamodule.dataset_params.dentate_gyrus.sd_size_factor={self.sd_pkl}",
        
        #     # what you actually want to control at generation time
        #     f"datamodule.datamodule.test_batch_size={tb}",
        #     f"++model.module.generation_args.timesteps={ts}",
        #     f"++model.module.generation_args.guidance_weight={{{self.label_key}:{gw}}}",
        
        #     # avoid the interpolation crash path
        #     "++datamodule.dataset_params.homo_sapiens.metadata_genes=null",
        # ]
        
        self._run(cmd_inf, cwd=self.scldm_root)
    
        out_h5ad = os.path.join(self.infer_dir, f"dentate_gyrus_generated_{int(gen_idx)}.h5ad")
        if not os.path.exists(out_h5ad):
            raise FileNotFoundError("scLDM inference did not produce: " + out_h5ad)
    
        adata_gen = ad.read_h5ad(out_h5ad)
    
        # select unconditional or conditional half
        if "dataset" in adata_gen.obs:
            key = "generated_unconditional" if unconditional else "generated_conditional"
            ad_sel = adata_gen[adata_gen.obs["dataset"].values == key].copy()
            if ad_sel.n_obs == 0:
                # fallback: split by halves
                n = adata_gen.n_obs // 2
                ad_sel = adata_gen[:n].copy() if unconditional else adata_gen[n:].copy()
        else:
            n = adata_gen.n_obs // 2
            ad_sel = adata_gen[:n].copy() if unconditional else adata_gen[n:].copy()

        target = self._canonical_var_names()
        cur = np.asarray(ad_sel.var_names, dtype=str)
        
        if (len(cur) != len(target)) or (not np.array_equal(cur, target)):
            tkey = self._dup_keys(target)
            ckey = self._dup_keys(cur)
        
            pos = {k: i for i, k in enumerate(ckey)}
            idx = np.fromiter((pos.get(k, -1) for k in tkey), dtype=np.int64, count=len(tkey))
        
            if (idx < 0).any():
                missing = tkey[idx < 0][:10]
                raise RuntimeError(f"[scLDM] generated output missing columns (first 10 keys): {missing.tolist()}")
        
            ad_sel = ad_sel[:, idx].copy()
        
        Xg = ad_sel.X
        if hasattr(Xg, "toarray"):
            Xg = Xg.toarray()
        Xg = np.asarray(Xg)
        if not np.issubdtype(Xg.dtype, np.integer):
            Xg = np.rint(Xg).astype(np.int64)
        return Xg
    
    
    def sample(self, n_samples, n_steps=None, batch_size=None, unconditional=True):
        n = int(n_samples)
        ts = self.timesteps if (n_steps is None) else int(n_steps)
        tb = self.test_batch_size if (batch_size is None) else int(batch_size)
    
        chunks = []
        got = 0
        gen_idx = 0
        while got < n:
            Xg = self._infer_one_round(
                gen_idx=gen_idx,
                timesteps=ts,
                test_batch_size=tb,
                unconditional=bool(unconditional),
            )
            chunks.append(Xg)
            got += Xg.shape[0]
            gen_idx += 1
        X = np.concatenate(chunks, axis=0)[:n]
        return X

## 3.6 DCM

In [17]:
# !git clone https://github.com/sanjukta7/aivc-dcm aivc-dcm

In [18]:
# -------------------------
# 3.x DCM (Discrete Cell Models)
# -------------------------
import os, json, subprocess
from pathlib import Path
import numpy as np
import anndata as ad

class DCMModel(UncondModel):
    name = "DCM"

    def __init__(
        self,
        dcm_root="aivc-dcm",
        py_exec=None,                 # python>=3.10 recommended by the repo
        workdir="saved_models/dcm_dg",
        seed=0,
        device="cuda",
        # training
        hidden_dim=128,
        num_layers=4,
        num_heads=4,
        dropout=0.1,
        batch_size=32,
        num_epochs=50,
        learning_rate=1e-4,
        weight_decay=1e-2,
        mask_ratio=0.15,
        val_fraction=0.1,
        num_workers=0,
        save_interval=2,
        # sampling
        default_num_steps=50,
        default_temperature=1.0,
        # important: reserve a dedicated mask token (fixes repo's NUM_BINS=max(data) collision)
        apply_mask_token_fix=True,
    ):
        # self.dcm_root = str(dcm_root)
        self.dcm_root = str(Path(dcm_root).expanduser().resolve())
        self.py_exec = py_exec if py_exec is not None else sys.executable
        # self.workdir = str(workdir)
        self.workdir  = str(Path(workdir).expanduser().resolve())
        self.seed = int(seed)
        self.device = str(device)

        self.hidden_dim = int(hidden_dim)
        self.num_layers = int(num_layers)
        self.num_heads = int(num_heads)
        self.dropout = float(dropout)

        self.batch_size = int(batch_size)
        self.num_epochs = int(num_epochs)
        self.learning_rate = float(learning_rate)
        self.weight_decay = float(weight_decay)
        self.mask_ratio = float(mask_ratio)
        self.val_fraction = float(val_fraction)
        self.num_workers = int(num_workers)
        self.save_interval = int(save_interval)

        self.default_num_steps = int(default_num_steps)
        self.default_temperature = float(default_temperature)

        self.apply_mask_token_fix = bool(apply_mask_token_fix)

        # filled after fit/load
        self.train_h5ad = None
        self.ckpt_dir = None
        self.mask_token = None   # integer token reserved for masking

    def _repo_path(self):
        p = Path(self.dcm_root)
        if not p.exists():
            raise FileNotFoundError(f"DCM repo not found at: {p}")
        return p

    def _scripts_dir(self):
        p = self._repo_path() / "scripts"
        if not p.exists():
            raise FileNotFoundError(f"DCM scripts/ not found under: {p}")
        return p

    def _subproc_env(self):
        env = os.environ.copy()
        repo = str(self._repo_path().resolve())
        src  = str((self._repo_path() / "src").resolve())
    
        parts = [src, repo]
        old = env.get("PYTHONPATH", "")
        if old:
            parts.append(old)
    
        env["PYTHONPATH"] = os.pathsep.join(parts)
        env["PYTORCH_NVML_BASED_CUDA_CHECK"] = "0"
        env["CUDA_DEVICE_ORDER"] = "PCI_BUS_ID"
        return env

    def _assert_cuda_in_subproc(self):
        cmd = [self.py_exec, "-c", "import torch; print(int(torch.cuda.is_available()), torch.cuda.device_count())"]
        r = subprocess.run(cmd, env=self._subproc_env(), capture_output=True, text=True)
        out = (r.stdout + "\n" + r.stderr).strip()
        print("DCM cuda check:", out)
        if r.returncode != 0 or out.startswith("0 "):
            raise RuntimeError("DCM subprocess has no CUDA. Run this notebook on a GPU node (nvidia-smi must work) or use a CUDA-enabled torch env.")        

        

    def _ensure_maskfix_scripts(self):
        """
        The upstream scripts set:
            NUM_BINS = int(dataset.max().item())
            VOCAB_SIZE = NUM_BINS + 1  # +1 for mask token
        But model code uses mask_index=num_bins, so mask token collides with max count.
        Fix: NUM_BINS = max_count + 1, so mask token is a dedicated extra state.
        We do this by writing patched copies inside repo/scripts so sys.path logic still works.
        """
        scripts = self._scripts_dir()
        train_src = scripts / "train_rnaseq.py"
        infer_src = scripts / "infernce_generation.py"  # repo spelling

        train_dst = scripts / "train_rnaseq_maskfix.py"
        infer_dst = scripts / "infernce_generation_maskfix.py"

        def _patch_one(src, dst):
            if dst.exists():
                return str(dst)
            txt = src.read_text()
            needle = "NUM_BINS = int(dataset.max().item())"
            if needle not in txt:
                raise RuntimeError(f"Expected pattern not found in {src.name}: {needle}")
            txt2 = txt.replace(needle, "NUM_BINS = int(dataset.max().item()) + 1", 1)
            dst.write_text(txt2)
            return str(dst)

        train_script = str(train_src)
        infer_script = str(infer_src)
        if self.apply_mask_token_fix:
            train_script = _patch_one(train_src, train_dst)
            infer_script = _patch_one(infer_src, infer_dst)

        # return train_script, infer_script
        return str(Path(train_script).resolve()), str(Path(infer_script).resolve())

    def _write_train_h5ad(self, X_train_counts, var_names=None):
        Path(self.workdir).mkdir(parents=True, exist_ok=True)
        out = Path(self.workdir) / "dcm_train_counts.h5ad"

        X = np.asarray(X_train_counts)
        if not np.issubdtype(X.dtype, np.integer):
            X = np.rint(X).astype(np.int64)

        # define the dedicated mask token we want the model to use
        self.mask_token = int(X.max()) + 1

        adata_tmp = ad.AnnData(X.astype(np.int32, copy=False))
        if var_names is not None:
            adata_tmp.var_names = np.asarray(var_names, dtype=str)

        # store metadata (not used by scripts, but useful for debugging)
        adata_tmp.uns["dcm_mask_token"] = int(self.mask_token)
        adata_tmp.uns["dcm_max_count"] = int(X.max())

        adata_tmp.write_h5ad(out)
        return str(out)

    def fit(self, X_train_counts, var_names=None, resume=True, **kwargs):
        # allow overrides
        num_epochs = int(kwargs.get("num_epochs", self.num_epochs))
        batch_size = int(kwargs.get("batch_size", self.batch_size))
        learning_rate = float(kwargs.get("learning_rate", self.learning_rate))
        weight_decay = float(kwargs.get("weight_decay", self.weight_decay))
        mask_ratio = float(kwargs.get("mask_ratio", self.mask_ratio))
        val_fraction = float(kwargs.get("val_fraction", self.val_fraction))

        self.train_h5ad = self._write_train_h5ad(X_train_counts, var_names=var_names)
        # self.ckpt_dir = str(Path(self.workdir) / "checkpoints")
        self.ckpt_dir = str((Path(self.workdir) / "checkpoints").resolve())
        Path(self.ckpt_dir).mkdir(parents=True, exist_ok=True)
        config_path = str((self._repo_path() / "configs" / "rnaseq_small.yaml").resolve())


        train_script, _ = self._ensure_maskfix_scripts()

        self._assert_cuda_in_subproc()
        cmd = [
            self.py_exec, train_script,
            # "--config", str(Path(self._repo_path()) / "configs" / "rnaseq_small.yaml"),
            # "--data_path", self.train_h5ad,
            # "--checkpoint_dir", self.ckpt_dir,

            "--config", config_path,
            "--data_path", str(Path(self.train_h5ad).resolve()),
            "--checkpoint_dir", self.ckpt_dir,
            
            "--seed", str(self.seed),

            "--hidden_dim", str(self.hidden_dim),
            "--num_layers", str(self.num_layers),
            "--num_heads", str(self.num_heads),
            "--dropout", str(self.dropout),

            "--batch_size", str(batch_size),
            "--num_epochs", str(num_epochs),
            "--learning_rate", str(learning_rate),
            "--weight_decay", str(weight_decay),
            "--mask_ratio", str(mask_ratio),
            "--val_fraction", str(val_fraction),
            "--num_workers", str(self.num_workers),
            "--save_interval", str(self.save_interval),
        ]
        if resume:
            cmd += ["--resume", "auto"]

        print("DCM train cmd:")
        print(" ".join(cmd))
        # subprocess.run(cmd, cwd=str(self._repo_path()), check=True)
        subprocess.run(cmd, cwd=str(self._repo_path()), check=True, env=self._subproc_env())
        print("DCM training done. ckpt_dir =", self.ckpt_dir)


    def sample(self, n_samples, n_steps=None, temperature=None, gen_chunk=16, **kwargs):
        if self.ckpt_dir is None or self.train_h5ad is None:
            raise RuntimeError("DCMModel.sample called before fit/load.")
    
        n_samples = int(n_samples)
        gen_chunk = int(gen_chunk)
        if gen_chunk <= 0:
            raise ValueError("gen_chunk must be positive")
    
        n_steps = self.default_num_steps if n_steps is None else int(n_steps)
        temperature = self.default_temperature if temperature is None else float(temperature)
    
        _, infer_script = self._ensure_maskfix_scripts()
    
        # helps fragmentation a bit, harmless otherwise
        env = self._subproc_env()
        env.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
    
        outs = []
        done = 0
        chunk_id = 0
    
        while done < n_samples:
            cur = min(gen_chunk, n_samples - done)
    
            cmd = [
                self.py_exec, infer_script,
                "--experiment_dir", str(Path(self.ckpt_dir).resolve()),
                "--data_path", str(Path(self.train_h5ad).resolve()),
                "--num_generate", str(cur),
                "--num_steps", str(n_steps),
                "--temperature", str(temperature),
                "--seed", str(int(self.seed) + chunk_id),   # avoid identical chunks
                "--val_fraction", str(self.val_fraction),
                "--use_train_split",
            ]
    
            print(f"DCM sample cmd (chunk {chunk_id}, n={cur}):")
            print(" ".join(cmd))
    
            subprocess.run(cmd, cwd=str(self._repo_path()), check=True, env=env)
    
            out_dir = Path(self.ckpt_dir) / "generation_results"
            npy_path = out_dir / "generated_cells.npy"
            if not npy_path.exists():
                raise FileNotFoundError(f"DCM generation output not found: {npy_path}")
    
            Xg = np.load(npy_path)
            Xg = np.asarray(Xg, dtype=np.int64)
            outs.append(Xg)
    
            done += cur
            chunk_id += 1
    
        X = np.concatenate(outs, axis=0)
    
        # map any leftover mask tokens to 0 (safe)
        if self.mask_token is None:
            self.mask_token = int(np.max(X))
        X[X == int(self.mask_token)] = 0
    
        return X

## 3.7 save and load

In [19]:
import os, json, torch

def save_countfm(model, path_dir):
    os.makedirs(path_dir, exist_ok=True)

    payload = {
        "state_dict": model.net.state_dict(),
        "config": {
            "d": model.d,
            "eps_t": model.eps_t,
            "eps_log": model.eps_log,
            "loss_mode": model.loss_mode,
            "margin": model.margin,
            "C_max": model.C_max,
            "x0_mode_train": model.x0_mode_train,
            "ot_cost": model.ot_cost,
            "d_model": model.d_model,
            "depth": model.depth,
            "n_heads": model.n_heads,
            "mlp_ratio": model.mlp_ratio,
            "dropout": model.dropout,
            "chunk_size": model.chunk_size,
            "log_cap": model.log_cap,
        }
    }
    torch.save(payload, os.path.join(path_dir, "countfm.pt"))
    print("Saved count-FM to", path_dir)

import torch

def load_countfm(path_dir, device="cuda"):
    ckpt = torch.load(os.path.join(path_dir, "countfm.pt"), map_location="cpu")
    cfg = ckpt["config"]
    d = int(cfg["d"])

    dev = torch.device(device if (device == "cpu" or torch.cuda.is_available()) else "cpu")

    # rebuild exact net
    net_base = models.ChunkedAdaLNTransformer(
        dim=d,
        out_dim=2 * d,
        time_varying=True,
        d_model=int(cfg["d_model"]),
        depth=int(cfg["depth"]),
        n_heads=int(cfg["n_heads"]),
        mlp_ratio=float(cfg["mlp_ratio"]),
        dropout=float(cfg["dropout"]),
        chunk_size=int(cfg["chunk_size"]),
    ).to(dev)

    net = models.ChunkedAdaLNTransformer_rate(
        net_base,
        log_cap=float(cfg["log_cap"]),
        eps=float(cfg["eps_log"]),
    ).to(dev)

    net.load_state_dict(ckpt["state_dict"])

    m = CountFMModel(
        device=device,
        eps_t=float(cfg["eps_t"]),
        eps_log=float(cfg["eps_log"]),
        loss_mode=str(cfg["loss_mode"]),
        margin=float(cfg["margin"]),
        C_max=cfg["C_max"],
        x0_mode_train=str(cfg["x0_mode_train"]),
        ot_cost=str(cfg["ot_cost"]),
        d_model=int(cfg["d_model"]),
        depth=int(cfg["depth"]),
        n_heads=int(cfg["n_heads"]),
        mlp_ratio=float(cfg["mlp_ratio"]),
        dropout=float(cfg["dropout"]),
        chunk_size=int(cfg["chunk_size"]),
        log_cap=float(cfg["log_cap"]),
    )
    m.net = net
    m.d = d
    print("Loaded count-FM from", path_dir)
    return m

In [20]:
import os
import numpy as np
import anndata as ad
import scvi
import torch

def save_scvi(scvi_wrapper, path_dir):
    os.makedirs(path_dir, exist_ok=True)
    # critical: save AnnData so SCVI.load(path_dir) works
    scvi_wrapper.model.save(path_dir, overwrite=True, save_anndata=True)
    with open(os.path.join(path_dir, "scvi_tools_version.txt"), "w") as f:
        f.write(scvi.__version__)
    print("Saved scVI to", path_dir, "(save_anndata=True)")

def load_scvi(path_dir, device="cuda", X_ref_counts=None):
    """
    PyTorch 2.6: torch.load defaults weights_only=True, which breaks scvi-tools loading.
    Since this is YOUR checkpoint, we force weights_only=False when needed.
    """
    def _do_load():
        try:
            return scvi.model.SCVI.load(path_dir)
        except Exception as e:
            msg = str(e).lower()
            if ("anndata" in msg) or ("adata" in msg):
                if X_ref_counts is None:
                    raise RuntimeError(
                        "SCVI.load failed because no AnnData was saved. "
                        "Pass X_ref_counts once, then re-save with save_anndata=True."
                    ) from e
                adata_tmp = ad.AnnData(np.asarray(X_ref_counts).astype(np.int64))
                return scvi.model.SCVI.load(path_dir, adata=adata_tmp)
            raise

    try:
        model = _do_load()
    except Exception as e:
        # PyTorch 2.6 weights_only failure: retry with weights_only=False (trusted checkpoint)
        if ("weights only load failed" in str(e).lower()) or ("weights_only" in str(e).lower()):
            _torch_load = torch.load
            def _torch_load_force(*args, **kwargs):
                kwargs.setdefault("weights_only", False)
                return _torch_load(*args, **kwargs)
            torch.load = _torch_load_force
            try:
                model = _do_load()
            finally:
                torch.load = _torch_load
        else:
            raise

    # move to device
    if device == "cuda" and torch.cuda.is_available():
        model.to_device("cuda:0")
    else:
        model.to_device("cpu")

    wrapper = SCVIModel(device=device)
    wrapper.model = model
    print("Loaded scVI from", path_dir)
    return wrapper

In [21]:
import os, json, numpy as np

def save_scdiffusion_bundle(scdiff_model, bundle_dir):
    os.makedirs(bundle_dir, exist_ok=True)
    meta = dict(
        scd_root=scdiff_model.scd_root,
        workdir=scdiff_model.workdir,
        seed=scdiff_model.seed,
        ae_steps=scdiff_model.ae_steps,
        diff_steps=scdiff_model.diff_steps,
        batch_size=scdiff_model.batch_size,
        ae_ckpt_freq=scdiff_model.ae_ckpt_freq,
        diff_save_interval=scdiff_model.diff_save_interval,
        lr=scdiff_model.lr,
        weight_decay=scdiff_model.weight_decay,
        latent_dim=scdiff_model.latent_dim,
        hidden_dim=scdiff_model.hidden_dim,
        noise_schedule=scdiff_model.noise_schedule,
        diffusion_steps=scdiff_model.diffusion_steps,

        # new fields (gene filtering + padding)
        d_full=int(scdiff_model.d_full) if scdiff_model.d_full is not None else None,
        d_filtered=int(scdiff_model.d) if scdiff_model.d is not None else None,
        keep_gene_idx=(scdiff_model.keep_gene_idx.tolist() if scdiff_model.keep_gene_idx is not None else None),
        keep_cell_idx=(scdiff_model.keep_cell_idx.tolist() if scdiff_model.keep_cell_idx is not None else None),

        data_h5ad=scdiff_model.data_h5ad,
        ae_ckpt=scdiff_model.ae_ckpt,
        backbone_ckpt=scdiff_model.backbone_ckpt,
    )
    with open(os.path.join(bundle_dir, "scdiffusion_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)

    np.savez_compressed(
        os.path.join(bundle_dir, "scdiffusion_stats.npz"),
        train_lib_sizes=scdiff_model._train_lib_sizes.astype(np.int64),
    )
    print("Saved scDiffusion bundle:", bundle_dir)

def load_scdiffusion_bundle(bundle_dir, device="cuda"):
    meta = json.load(open(os.path.join(bundle_dir, "scdiffusion_meta.json"), "r"))
    stats = np.load(os.path.join(bundle_dir, "scdiffusion_stats.npz"))

    m = ScDiffusionModel(
        scd_root=meta["scd_root"],
        workdir=meta["workdir"],
        scimilarity_dir=None,  # not needed for sampling
        device=device,
        seed=meta["seed"],
        ae_steps=meta["ae_steps"],
        diff_steps=meta["diff_steps"],
        batch_size=meta["batch_size"],
        ae_ckpt_freq=meta["ae_ckpt_freq"],
        diff_save_interval=meta["diff_save_interval"],
        lr=meta["lr"],
        weight_decay=meta["weight_decay"],
        latent_dim=meta["latent_dim"],
        hidden_dim=meta["hidden_dim"],
        noise_schedule=meta["noise_schedule"],
        diffusion_steps=meta["diffusion_steps"],
    )

    # new fields
    m.d_full = meta.get("d_full", None)
    m.d = meta.get("d_filtered", meta.get("d", None))
    kg = meta.get("keep_gene_idx", None)
    kc = meta.get("keep_cell_idx", None)
    m.keep_gene_idx = None if kg is None else np.array(kg, dtype=int)
    m.keep_cell_idx = None if kc is None else np.array(kc, dtype=int)

    m.data_h5ad = meta["data_h5ad"]
    m.ae_ckpt = meta["ae_ckpt"]
    m.backbone_ckpt = meta["backbone_ckpt"]
    m._train_lib_sizes = stats["train_lib_sizes"].astype(np.int64)

    print("Loaded scDiffusion bundle:", bundle_dir)
    return m

In [22]:
import os, json, re
import numpy as np
import torch

from cfgen.models.base.encoder_model import EncoderModel
from cfgen.models.featurizers.category_featurizer import CategoricalFeaturizer
from cfgen.models.fm.denoising_model import MLPTimeStep
from cfgen.models.fm.fm import FM


def _jsonify(x):
    if isinstance(x, dict):
        return {str(k): _jsonify(v) for k, v in x.items()}
    if isinstance(x, (list, tuple)):
        return [_jsonify(v) for v in x]
    if isinstance(x, (set,)):
        return [_jsonify(v) for v in sorted(list(x))]
    if isinstance(x, (np.integer,)):
        return int(x)
    if isinstance(x, (np.floating,)):
        return float(x)
    if isinstance(x, (np.bool_,)):
        return bool(x)
    if isinstance(x, np.ndarray):
        return x.tolist()
    if torch.is_tensor(x):
        return x.detach().cpu().tolist()
    return x


def _torch_load_weights(path, map_location="cpu"):
    # Works on torch<2.6 and torch>=2.6 (weights-only tensors)
    try:
        return torch.load(path, map_location=map_location, weights_only=True)
    except TypeError:
        return torch.load(path, map_location=map_location)


def _infer_encoder_dims_and_d_from_sd(sd):
    pat = re.compile(r"^encoder\.rna\.net\.(\d+)(?:\.(\d+))?\.weight$")
    layers = {}
    for k, v in sd.items():
        m = pat.match(k)
        if m and isinstance(v, torch.Tensor) and v.ndim == 2:
            i = int(m.group(1))
            layers.setdefault(i, (k, v))
    idx = sorted(layers.keys())
    if not idx:
        raise RuntimeError("Cannot infer encoder dims from encoder state_dict keys.")
    d = int(layers[idx[0]][1].shape[1])
    dims = [int(layers[i][1].shape[0]) for i in idx]
    return d, dims


def _remap_keys_to_match_target(src_sd, tgt_keys):
    out = {}
    for k, v in src_sd.items():
        if k in tgt_keys:
            out[k] = v
            continue
        k2 = re.sub(r"\.net\.(\d+)\.weight$", r".net.\1.0.weight", k)
        if k2 != k and k2 in tgt_keys:
            out[k2] = v
            continue
        k2 = re.sub(r"\.net\.(\d+)\.bias$", r".net.\1.0.bias", k)
        if k2 != k and k2 in tgt_keys:
            out[k2] = v
            continue
        k2 = re.sub(r"\.net\.(\d+)\.0\.weight$", r".net.\1.weight", k)
        if k2 != k and k2 in tgt_keys:
            out[k2] = v
            continue
        k2 = re.sub(r"\.net\.(\d+)\.0\.bias$", r".net.\1.bias", k)
        if k2 != k and k2 in tgt_keys:
            out[k2] = v
            continue
    return out


def save_cfgen_bundle(cfgen_model, bundle_dir):
    # IMPORTANT: save from in-memory models (NO loading .ckpt -> no pickle/safe_globals issues)
    assert cfgen_model.encoder_model is not None and cfgen_model.fm_model is not None, \
        "CFGen not trained/loaded in memory: encoder_model or fm_model is None."
    assert cfgen_model._meta is not None, "CFGen has no _meta. Call fit() first."

    os.makedirs(bundle_dir, exist_ok=True)

    torch.save(cfgen_model.encoder_model.state_dict(),
               os.path.join(bundle_dir, "cfgen_encoder_sd.pt"))
    torch.save(cfgen_model.fm_model.state_dict(),
               os.path.join(bundle_dir, "cfgen_fm_sd.pt"))

    with open(os.path.join(bundle_dir, "cfgen_meta.json"), "w") as f:
        json.dump(_jsonify(cfgen_model._meta), f, indent=2)

    print("Saved CFGen bundle (weights-only):", bundle_dir)


def load_cfgen_bundle(bundle_dir, device="cuda"):
    meta = json.load(open(os.path.join(bundle_dir, "cfgen_meta.json"), "r"))
    enc_path = os.path.join(bundle_dir, "cfgen_encoder_sd.pt")
    fm_path  = os.path.join(bundle_dir, "cfgen_fm_sd.pt")

    dev = torch.device("cuda" if (device == "cuda" and torch.cuda.is_available()) else "cpu")

    def _as_float_rna(v):
        while True:
            if isinstance(v, dict):
                v = v["rna"] if ("rna" in v) else next(iter(v.values()))
                continue
            if isinstance(v, (list, tuple, np.ndarray)):
                v = v[0]
                continue
            if torch.is_tensor(v):
                v = v.detach().cpu().flatten()[0].item()
                continue
            return float(v)

    # ---------- load weights-only ----------
    enc_sd = _torch_load_weights(enc_path, map_location="cpu")
    fm_sd  = _torch_load_weights(fm_path,  map_location="cpu")

    # ---------- rebuild encoder from its state_dict ----------
    d_ckpt, dims_ckpt = _infer_encoder_dims_and_d_from_sd(enc_sd)
    zdim = int(dims_ckpt[-1])
    has_bn = any(".running_mean" in k for k in enc_sd.keys()) or any(".running_var" in k for k in enc_sd.keys())

    encoder_kwargs = {"rna": {"dims": list(dims_ckpt), "batch_norm": bool(has_bn), "dropout": False, "dropout_p": 0.0}}
    enc = EncoderModel(
        in_dim={"rna": d_ckpt},
        encoder_kwargs=encoder_kwargs,
        learning_rate=1e-3,
        weight_decay=1e-5,
        covariate_specific_theta=False,
        conditioning_covariate=meta["covariate_key"],
        n_cat=None,
        is_binarized=False,
    )

    tgt_keys = set(enc.state_dict().keys())
    enc_sd2 = _remap_keys_to_match_target(enc_sd, tgt_keys)
    enc.load_state_dict(enc_sd2, strict=False)
    enc.eval()
    for p in enc.parameters():
        p.requires_grad = False
    enc = enc.to(dev)

    # ---------- rebuild FM ----------
    cov = meta["covariate_key"]
    fm_params = meta["fm_params"]
    n_cat = len(meta["id2cov"][cov])

    feat = CategoricalFeaturizer(
        n_cat=n_cat,
        one_hot_encode_features=bool(fm_params["one_hot_encode_features"]),
        device=dev,
        embedding_dimensions=int(fm_params["embedding_dim"]),
    )
    feature_embeddings = {cov: feat}

    denoise = MLPTimeStep(
        in_dim=zdim,
        hidden_dim=int(fm_params["hidden_dim"]),
        dropout_prob=float(fm_params["dropout_prob"]),
        n_blocks=int(fm_params["n_blocks"]),
        size_factor_min=_as_float_rna(meta["min_size_factor"]),
        size_factor_max=_as_float_rna(meta["max_size_factor"]),
        embed_size_factor=bool(fm_params["embed_size_factor"]),
        covariate_list=[cov],
        embedding_dim=int(fm_params["embedding_dim"]),
        normalization=str(fm_params["normalization"]),
        conditional=bool(fm_params["conditional"]),
        is_binarized=False,
        modality_list=["rna"],
        conditioning_probability=float(fm_params["conditioning_probability"]),
        guided_conditioning=bool(fm_params["guided_conditioning"]),
    ).to(dev)

    sf = meta["size_factor_statistics"]
    fm = FM(
        encoder_model=enc,
        denoising_model=denoise,
        feature_embeddings=feature_embeddings,
        plotting_folder=None,
        in_dim={"rna": zdim},
        size_factor_statistics={"mean": _as_float_rna(sf["mean"]), "sd": _as_float_rna(sf["sd"])},
        covariate_list=[cov],
        theta_covariate=cov,
        size_factor_covariate=cov,
        encoder_type="fixed",
        learning_rate=float(fm_params["lr"]),
        weight_decay=float(fm_params["wd"]),
        antithetic_time_sampling=bool(fm_params["antithetic_time_sampling"]),
        sigma=float(fm_params["sigma"]),
        covariate_specific_theta=False,
        plot_and_eval_every=10**9,
        use_ot=bool(fm_params["use_ot"]),
        is_binarized=False,
        modality_list=["rna"],
        guidance_weights={cov: 1},
    ).to(dev)

    # don’t force strict=True (CFGen versions differ); also ignore encoder weights here
    fm_sd2 = {k: v for k, v in fm_sd.items() if not k.startswith("encoder_model.")}
    keys = fm.load_state_dict(fm_sd2, strict=False)
    fm.eval()

    m = CFGenModel(device=device, workdir="__loaded__", covariate_key=cov)
    m.encoder_model = enc
    m.fm_model = fm
    m._meta = meta

    print("Loaded CFGen bundle (weights-only):", bundle_dir, "| zdim =", zdim,
          "| missing =", len(keys.missing_keys), "| unexpected =", len(keys.unexpected_keys))
    return m

In [23]:
def save_scldm_bundle(scldm_model, bundle_dir):
    os.makedirs(bundle_dir, exist_ok=True)
    meta = dict(
        scldm_root=scldm_model.scldm_root,
        py_exec=scldm_model.py_exec,
        workdir=scldm_model.workdir,
        seed=scldm_model.seed,
        devices=getattr(scldm_model, "devices", 1),
        vae_epochs=scldm_model.vae_epochs,
        ldm_epochs=scldm_model.ldm_epochs,
        batch_size=scldm_model.batch_size,
        test_batch_size=scldm_model.test_batch_size,
        num_workers=scldm_model.num_workers,
        timesteps=scldm_model.timesteps,
        guidance_weight=scldm_model.guidance_weight,
        d=scldm_model.d,
        var_names=scldm_model.var_names,
        train_h5ad=scldm_model.train_h5ad,
        test_h5ad=scldm_model.test_h5ad,
        mu_pkl=scldm_model.mu_pkl,
        sd_pkl=scldm_model.sd_pkl,
        vae_name=scldm_model.vae_name,
        ldm_name=scldm_model.ldm_name,
        ckpt_root=scldm_model.ckpt_root,
    )
    with open(os.path.join(bundle_dir, "scldm_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print("Saved scLDM bundle:", bundle_dir)

def load_scldm_bundle(bundle_dir):
    meta = json.load(open(os.path.join(bundle_dir, "scldm_meta.json"), "r"))
    m = SCLDMModel(
        scldm_root=meta["scldm_root"],
        py_exec=meta["py_exec"],
        workdir=meta["workdir"],
        seed=meta["seed"],
        devices=meta.get("devices", 1),
        vae_epochs=meta["vae_epochs"],
        ldm_epochs=meta["ldm_epochs"],
        batch_size=meta["batch_size"],
        test_batch_size=meta["test_batch_size"],
        num_workers=meta["num_workers"],
        timesteps=meta["timesteps"],
        guidance_weight=meta["guidance_weight"],
    )
    m.d = meta.get("d", None)
    m.var_names = meta.get("var_names", None)
    m.train_h5ad = meta.get("train_h5ad", None)
    m.test_h5ad = meta.get("test_h5ad", None)
    m.mu_pkl = meta.get("mu_pkl", None)
    m.sd_pkl = meta.get("sd_pkl", None)
    m.vae_name = meta.get("vae_name", None)
    m.ldm_name = meta.get("ldm_name", None)
    m.ckpt_root = meta.get("ckpt_root", None)
    if m.ckpt_root is not None and m.vae_name is not None:
        base_experiment_path = os.path.abspath(os.path.join(m.workdir, "experiments"))
        m.ckpt_root = os.path.join(base_experiment_path, "checkpoints")
        m.vae_ckpt_dir = os.path.join(m.ckpt_root, m.vae_name)
        m.ldm_ckpt_dir = os.path.join(m.ckpt_root, m.ldm_name)
    print("Loaded scLDM bundle:", bundle_dir)
    return m

In [24]:
import os, json

def save_dcm_bundle(dcm_model, bundle_dir):
    os.makedirs(bundle_dir, exist_ok=True)
    meta = dict(
        dcm_root=dcm_model.dcm_root,
        py_exec=dcm_model.py_exec,
        workdir=dcm_model.workdir,
        seed=dcm_model.seed,
        device=dcm_model.device,

        hidden_dim=dcm_model.hidden_dim,
        num_layers=dcm_model.num_layers,
        num_heads=dcm_model.num_heads,
        dropout=dcm_model.dropout,

        batch_size=dcm_model.batch_size,
        num_epochs=dcm_model.num_epochs,
        learning_rate=dcm_model.learning_rate,
        weight_decay=dcm_model.weight_decay,
        mask_ratio=dcm_model.mask_ratio,
        val_fraction=dcm_model.val_fraction,
        num_workers=dcm_model.num_workers,
        save_interval=dcm_model.save_interval,

        default_num_steps=dcm_model.default_num_steps,
        default_temperature=dcm_model.default_temperature,

        apply_mask_token_fix=dcm_model.apply_mask_token_fix,

        train_h5ad=dcm_model.train_h5ad,
        ckpt_dir=dcm_model.ckpt_dir,
        mask_token=(int(dcm_model.mask_token) if dcm_model.mask_token is not None else None),
    )
    with open(os.path.join(bundle_dir, "dcm_meta.json"), "w") as f:
        json.dump(meta, f, indent=2)
    print("Saved DCM bundle:", bundle_dir)

def load_dcm_bundle(bundle_dir):
    meta = json.load(open(os.path.join(bundle_dir, "dcm_meta.json"), "r"))
    m = DCMModel(
        dcm_root=meta["dcm_root"],
        py_exec=meta["py_exec"],
        workdir=meta["workdir"],
        seed=meta["seed"],
        device=meta["device"],

        hidden_dim=meta["hidden_dim"],
        num_layers=meta["num_layers"],
        num_heads=meta["num_heads"],
        dropout=meta["dropout"],

        batch_size=meta["batch_size"],
        num_epochs=meta["num_epochs"],
        learning_rate=meta["learning_rate"],
        weight_decay=meta["weight_decay"],
        mask_ratio=meta["mask_ratio"],
        val_fraction=meta["val_fraction"],
        num_workers=meta["num_workers"],
        save_interval=meta["save_interval"],

        default_num_steps=meta["default_num_steps"],
        default_temperature=meta["default_temperature"],
        apply_mask_token_fix=meta.get("apply_mask_token_fix", True),
    )
    m.train_h5ad = meta.get("train_h5ad", None)
    m.ckpt_dir   = meta.get("ckpt_dir", None)
    m.mask_token = meta.get("mask_token", None)
    return m

# 4. Training

## 4.1 Instantiate

In [25]:
# Instantiate
countfm_model = CountFMModel(
    device=DEVICE,
    num_epochs=500,
    batch_size=32,
    eps_t=1e-4,
    eps_log=1e-8,
    loss_mode="poisson",
    margin=2,
    C_max=None,

    # match 3_scRNA.ipynb behavior
    x0_mode_train="poisson",
    lam0_train=1.0,

    # OT coupling option (set "none" first; later try "sym_poisson"/"L2"/"L1")
    ot_cost="none",
    # ot_cost="sym_poisson",
    
    # transformer backbone (same family as 3_scRNA.ipynb)
    d_model=256,
    depth=8,
    n_heads=8,
    mlp_ratio=4.0,
    dropout=0.0,
    chunk_size=128,
    log_cap=50.0,
)

In [26]:
scvi_model = SCVIModel(
    device=DEVICE,
    n_latent=30,
    n_layers=3,
    n_hidden=256,
    gene_likelihood="zinb",
    max_epochs=800,
)

In [27]:
scdiff_model = ScDiffusionModel(
    scd_root="scDiffusion",
    workdir="saved_models/scdiffusion_dg_v1",
    scimilarity_dir=None,
    device=DEVICE,
    ae_steps=150000, # 150000
    diff_steps=600000, # 600000
    diff_save_interval=100000, # 100000
    batch_size=128,
)

In [28]:
cfgen_model = CFGenModel(
    device=DEVICE,
    workdir="saved_models/cfgen_dg_v1",
    seed=SEED,

    # unconditional setup
    covariate_key="dummy",
    fm_conditional=False,

    # you can cut epochs for quick tests
    encoder_max_epochs=500,
    fm_max_epochs=3000,
)

In [29]:
DIR_SCLDM = "saved_models/scldm_dg_v1"

SCLDM_PY   = "/home/ganchao/miniconda3/envs/vfm311/bin/python"
SCLDM_REPO = "/home/ganchao/Desktop/scLDM"

scldm_model = SCLDMModel(
    scldm_root=SCLDM_REPO ,
    py_exec=SCLDM_PY,
    workdir=DIR_SCLDM,
    seed=SEED,
    device="cuda",
    vae_epochs=100, # 100
    ldm_epochs=100, # 100
    batch_size=8,
    test_batch_size=16,
    num_workers=0,
    timesteps=100,
    guidance_weight=0.0,
)

In [30]:
DIR_DCM = "saved_models/dcm_dg_bundle_v1"

# Use a python>=3.10 interpreter if your notebook kernel is older.
# If your current kernel is already >=3.10 and has deps, you can set DCM_PY = None.
DCM_PY   = None
DCM_REPO = "aivc-dcm"

dcm_model = DCMModel(
    dcm_root=DCM_REPO,
    py_exec=DCM_PY,
    workdir="saved_models/dcm_dg_workdir_v1",
    seed=SEED,
    device=DEVICE,

    # defaults match rnaseq_small.yaml (small transformer)
    hidden_dim=256,
    num_layers=8,
    num_heads=8,
    dropout=0.1,

    batch_size=32,
    num_epochs=50,
    learning_rate=5e-4,
    weight_decay=0.0,
    mask_ratio=0.15,
    val_fraction=0.1,
    num_workers=0,
    save_interval=2,

    default_num_steps=100,
    default_temperature=1.0,

    apply_mask_token_fix=True,
)

## 4.2 fitting

### initial training

In [31]:
# !rm -rf saved_models/scldm_dg_v1/experiments/checkpoints/vae_dg_custom
# !rm -rf saved_models/scldm_dg_v1/experiments/checkpoints/ldm_dg_custom

In [32]:
# # count-FM (fit already accepts lr/wd/epochs)
# t0 = time.time()
# countfm_model.fit(X_train, lr=5e-4,
#                   weight_decay=1e-4, num_epochs=500, resume=True)
# print("count-FM trained in", round(time.time() - t0, 1), "sec")


# # scVI (fit already accepts lr/wd/max_epochs)
# t0 = time.time()
# scvi_model.fit(X_train, lr=5e-4,
#                weight_decay=1e-4, max_epochs=500, resume=True)
# print("scVI trained in", round(time.time() - t0, 1), "sec")


# # scDiffusion (fit accepts lr/wd, but steps/batch live on the object)
# scdiff_model.ae_steps = 150000
# scdiff_model.diff_steps = 600000
# scdiff_model.diff_save_interval = 100000
# scdiff_model.batch_size = 128

# t0 = time.time()
# scdiff_model.fit(X_train, lr=5e-4,
#                  weight_decay=1e-4, resume=True)
# print("scDiffusion trained in", round(time.time() - t0, 1), "sec")


# # CFGen (fit doesn’t expose optimizer/epochs, but the object has them)
# cfgen_model.encoder_max_epochs = 300
# cfgen_model.fm_max_epochs = 1500
# cfgen_model.encoder_lr = 1e-3
# cfgen_model.encoder_wd = 1e-5
# cfgen_model.fm_lr = 1e-4
# cfgen_model.fm_wd = 1e-6

# t0 = time.time()
# cfgen_model.fit(X_train, var_names=adata.var_names, resume=True)
# print("CFGen trained in", round(time.time() - t0, 1), "sec")


# # scLDM (fit doesn’t expose lr; you can still change budgets + sampler knobs)
# scldm_model.vae_epochs = 100
# scldm_model.ldm_epochs = 100
# scldm_model.batch_size = 8
# scldm_model.test_batch_size = 128
# scldm_model.timesteps = 100
# scldm_model.guidance_weight = 0.0

# t0 = time.time()
# scldm_model.fit(X_train, X_test, var_names=adata.var_names, resume=True)
# print("scLDM trained in", round(time.time() - t0, 1), "sec")


# # DCM (your DCMModel.fit already supports overriding these via kwargs)
# t0 = time.time()
# dcm_model.fit(
#     X_train,
#     var_names=adata.var_names,
#     resume=True,
#     batch_size=8,
#     num_epochs=50,
#     learning_rate=1e-4,
#     weight_decay=1e-2,
# )
# print("DCM trained in", round(time.time() - t0, 1), "sec")

### continue training

In [33]:
DIR_COUNTFM = "saved_models/countfm_dg_v1"
DIR_SCVI    = "saved_models/scvi_dg_v1"
DIR_SCD     = "saved_models/scdiffusion_dg_bundle_v1"
DIR_CFGEN   = "saved_models/cfgen_dg_bundle_v1"
DIR_SCLDM   = "saved_models/scldm_dg_v1"
DIR_DCM     = "saved_models/dcm_dg_bundle_v1"

In [34]:
countfm_model = load_countfm(DIR_COUNTFM, device=DEVICE)
try:
    scvi_model = load_scvi(DIR_SCVI, device=DEVICE)
except RuntimeError:
    scvi_model = load_scvi(DIR_SCVI, device=DEVICE, X_ref_counts=X_train)
scdiff_model = load_scdiffusion_bundle(DIR_SCD, device=DEVICE)
cfgen_model = load_cfgen_bundle(DIR_CFGEN, device=DEVICE)
scldm_model = load_scldm_bundle(DIR_SCLDM)
dcm_model = load_dcm_bundle(DIR_DCM)

Loaded count-FM from saved_models/countfm_dg_v1
INFO     File saved_models/scvi_dg_v1/model.pt already downloaded                                                  
INFO     File saved_models/scvi_dg_v1/model.pt already downloaded                                                  


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Loaded scVI from saved_models/scvi_dg_v1
Loaded scDiffusion bundle: saved_models/scdiffusion_dg_bundle_v1
Loaded CFGen bundle (weights-only): saved_models/cfgen_dg_bundle_v1 | zdim = 50 | missing = 29 | unexpected = 26
Loaded scLDM bundle: saved_models/scldm_dg_v1


/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'encoder_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder_model'])`.
/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'denoising_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['denoising_model'])`.


In [35]:
SCVI_EPOCHS      = 800

SCD_AE_STEPS     = 150000
SCD_DIFF_STEPS   = 600000
SCD_SAVE_INT     = 100000
SCD_BATCH        = 128

CFGEN_ENC_EPOCHS = 500
CFGEN_FM_EPOCHS  = 3000

SCLDM_VAE_EPOCHS = 200
SCLDM_LDM_EPOCHS = 200
SCLDM_BATCH      = 8
SCLDM_TEST_BATCH = 128

DCM_EPOCHS       = 100 + 1 - 72
DCM_BATCH        = 8
DCM_LR           = 5e-4
DCM_WD           = 0.0


# # count-FM (fit already accepts lr/wd/epochs)
# t0 = time.time()
# countfm_model.fit(X_train, lr=1e-3,
#                   weight_decay=1e-4, num_epochs=1000, resume=True)
# print("count-FM trained in", round(time.time() - t0, 1), "sec")



# # scVI
# t0 = time.time()
# scvi_model.fit(X_train, lr=2e-4, weight_decay=0.0,
#                max_epochs=SCVI_EPOCHS, resume=True)
# print("scVI refine:", round(time.time()-t0, 1), "sec")


# # scDiffusion (step-based, just raise targets then resume)
# scdiff_model.ae_steps = SCD_AE_STEPS
# scdiff_model.diff_steps = SCD_DIFF_STEPS
# scdiff_model.diff_save_interval = SCD_SAVE_INT
# scdiff_model.batch_size = SCD_BATCH

# t0 = time.time()
# scdiff_model.fit(X_train, lr=5e-4, weight_decay=0.0, resume=True)
# print("scDiff refine:", round(time.time()-t0, 1), "sec")


# # CFGen (Lightning resumes from last.ckpt; raise max_epochs)
# cfgen_model.encoder_max_epochs = CFGEN_ENC_EPOCHS
# cfgen_model.fm_max_epochs = CFGEN_FM_EPOCHS

# t0 = time.time()
# cfgen_model.fit(X_train, var_names=adata.var_names, resume=True)
# print("CFGen refine:", round(time.time()-t0, 1), "sec")


# # scLDM (Lightning resumes from last.ckpt; raise num_epochs)
# scldm_model.vae_epochs = SCLDM_VAE_EPOCHS
# scldm_model.ldm_epochs = SCLDM_LDM_EPOCHS
# scldm_model.batch_size = SCLDM_BATCH
# scldm_model.test_batch_size = SCLDM_TEST_BATCH



# import os, json, subprocess, re

# def _ckpt_epoch_via_py(py_exec, ckpt_path):
#     if not os.path.exists(ckpt_path):
#         return None
#     code = r"""
# import sys, torch
# p = sys.argv[1]
# ckpt = torch.load(p, map_location="cpu", weights_only=False)
# ep = ckpt.get("epoch", None)
# gs = ckpt.get("global_step", None)
# print(f"{ep} {gs}")
# """
#     r = subprocess.run([py_exec, "-c", code, ckpt_path], capture_output=True, text=True)
#     if r.returncode != 0:
#         print("ckpt read failed:", r.stderr[:300])
#         return None
#     m = re.findall(r"(-?\d+|None)", r.stdout.strip())
#     ep = None if (len(m) < 1 or m[0] == "None") else int(m[0])
#     return ep

# vae_last = os.path.join(scldm_model.workdir, "experiments", "checkpoints", "vae_dg_custom", "last.ckpt")
# ldm_last = os.path.join(scldm_model.workdir, "experiments", "checkpoints", "ldm_dg_custom", "last.ckpt")

# vae_ep = _ckpt_epoch_via_py(scldm_model.py_exec, vae_last)
# ldm_ep = _ckpt_epoch_via_py(scldm_model.py_exec, ldm_last)

# print("scLDM current epochs:", "VAE =", vae_ep, "| LDM =", ldm_ep)

# # continue +50 epochs from current
# if vae_ep is not None:
#     scldm_model.vae_epochs = max(int(scldm_model.vae_epochs), int(vae_ep) + 50)
# if ldm_ep is not None:
#     scldm_model.ldm_epochs = max(int(scldm_model.ldm_epochs), int(ldm_ep) + 50)

# print("scLDM new targets:", "VAE =", scldm_model.vae_epochs, "| LDM =", scldm_model.ldm_epochs)


# t0 = time.time()
# scldm_model.fit(X_train, X_test, var_names=adata.var_names, resume=True)
# print("scLDM refine:", round(time.time()-t0, 1), "sec")


# # DCM (script auto-resumes; raise num_epochs)
# t0 = time.time()
# dcm_model.fit(X_train, var_names=adata.var_names, resume=True,
#               batch_size=DCM_BATCH, num_epochs=DCM_EPOCHS,
#               learning_rate=DCM_LR, weight_decay=DCM_WD)
# print("DCM refine:", round(time.time()-t0, 1), "sec")

## 4.3 Save and Load

In [36]:
DIR_COUNTFM = "saved_models/backup/countfm_dg"
DIR_SCVI    = "saved_models/scvi_dg_v1"
DIR_SCD     = "saved_models/scdiffusion_dg_bundle_v1"
DIR_CFGEN   = "saved_models/cfgen_dg_bundle_v1"
DIR_SCLDM   = "saved_models/scldm_dg_v1"
DIR_DCM     = "saved_models/dcm_dg_bundle_v1"

In [92]:
# save_countfm(countfm_model, DIR_COUNTFM)
# save_scvi(scvi_model, DIR_SCVI)
# save_scdiffusion_bundle(scdiff_model, DIR_SCD)
# save_cfgen_bundle(cfgen_model, DIR_CFGEN)
# save_scldm_bundle(scldm_model, DIR_SCLDM)
# save_dcm_bundle(dcm_model, DIR_DCM)

In [37]:
countfm_model = load_countfm(DIR_COUNTFM, device=DEVICE)

Loaded count-FM from saved_models/backup/countfm_dg


In [71]:
# try:
#     scvi_model = load_scvi(DIR_SCVI, device=DEVICE)
# except RuntimeError:
#     scvi_model = load_scvi(DIR_SCVI, device=DEVICE, X_ref_counts=X_train)

INFO     File saved_models/scvi_dg_v1/model.pt already downloaded                                                  
INFO     File saved_models/scvi_dg_v1/model.pt already downloaded                                                  


An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


Loaded scVI from saved_models/scvi_dg_v1


In [72]:
# scdiff_model = load_scdiffusion_bundle(DIR_SCD, device=DEVICE)

Loaded scDiffusion bundle: saved_models/scdiffusion_dg_bundle_v1


In [73]:
# cfgen_model = load_cfgen_bundle(DIR_CFGEN, device=DEVICE)

Loaded CFGen bundle (weights-only): saved_models/cfgen_dg_bundle_v1 | zdim = 50 | missing = 29 | unexpected = 0


/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'encoder_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['encoder_model'])`.
/home/ganchao/miniconda3/envs/vfm/lib/python3.9/site-packages/pytorch_lightning/utilities/parsing.py:210: Attribute 'denoising_model' is an instance of `nn.Module` and is already saved during checkpointing. It is recommended to ignore them using `self.save_hyperparameters(ignore=['denoising_model'])`.


In [74]:
# scldm_model = load_scldm_bundle(DIR_SCLDM)

Loaded scLDM bundle: saved_models/scldm_dg_v1


In [75]:
# dcm_model = load_dcm_bundle(DIR_DCM)

# 5. Generation

In [38]:
COUNT_FM_LAM0 = 1.0
COUNT_FM_NSTEP = 500
COUNT_FM_BATCH_EVAL = 32
CFGEN_NSTEP = 50
CFGEN_BATCH = 512
SCLDM_NSTEP = 100
SCLDM_BATCH = 128
DCM_NSTEP = 100
DCM_TEMP  = 1.0

In [39]:
n_gen = X_test.shape[0] if (N_GEN is None) else int(N_GEN)
print("Generating n_gen =", n_gen)

Generating n_gen = 586


In [40]:
# count-FM
Xhat_countfm = countfm_model.sample(
    n_samples=n_gen,
    n_step=COUNT_FM_NSTEP,
    batch_eval=COUNT_FM_BATCH_EVAL,
    x0_mode="poisson",
    lam0=1.0,
)

# scVI
Xhat_scvi = scvi_model.sample(n_samples=n_gen)

# scDiffusion
Xhat_scd = scdiff_model.sample(n_samples=n_gen, batch_size=1000)

# CFGen
Xhat_cfgen = cfgen_model.sample(
    n_samples=n_gen,
    n_steps=CFGEN_NSTEP,
    batch_size=CFGEN_BATCH,
    unconditional=True,
)

# scLDM
Xhat_scldm = scldm_model.sample(
    n_samples=n_gen,
    n_steps=SCLDM_NSTEP,
    batch_size=SCLDM_BATCH,
    unconditional=True,
)

# DCM
Xhat_dcm = dcm_model.sample(
    n_samples=n_gen,
    n_steps=DCM_NSTEP,
    temperature=DCM_TEMP,
    gen_chunk=n_gen//10
)

def _cell_stats(Xc):
    lib = Xc.sum(axis=1)
    zfrac = (Xc == 0).mean(axis=1)
    return lib, zfrac

lib_r, z_r      = _cell_stats(X_test)
lib_cf, z_cf    = _cell_stats(Xhat_countfm)
lib_sv, z_sv    = _cell_stats(Xhat_scvi)
lib_scd, z_scd  = _cell_stats(Xhat_scd)
lib_cg, z_cg    = _cell_stats(Xhat_cfgen)
lib_scldm, z_scldm = _cell_stats(Xhat_scldm)
lib_dcm, z_dcm  = _cell_stats(Xhat_dcm)

print("count-FM:", Xhat_countfm.shape, "min/max:", Xhat_countfm.min(), Xhat_countfm.max(),
      "| lib(mean):", lib_cf.mean(), "zero_frac(mean):", z_cf.mean())
print("scVI:",    Xhat_scvi.shape,    "min/max:", Xhat_scvi.min(),    Xhat_scvi.max(),
      "| lib(mean):", lib_sv.mean(), "zero_frac(mean):", z_sv.mean())
print("scDiffusion:", Xhat_scd.shape, "min/max:", Xhat_scd.min(), Xhat_scd.max(),
      "| lib(mean):", lib_scd.mean(), "zero_frac(mean):", z_scd.mean())
print("CFGen:", Xhat_cfgen.shape, "min/max:", Xhat_cfgen.min(), Xhat_cfgen.max(),
      "| lib(mean):", lib_cg.mean(), "zero_frac(mean):", z_cg.mean())
print("scLDM:", Xhat_scldm.shape, "min/max:", Xhat_scldm.min(), Xhat_scldm.max(),
      "| lib(mean):", lib_scldm.mean(), "zero_frac(mean):", z_scldm.mean())
print("DCM:", Xhat_dcm.shape, "min/max:", Xhat_dcm.min(), Xhat_dcm.max(),
      "| lib(mean):", lib_dcm.mean(), "zero_frac(mean):", z_dcm.mean())
print("real test:", X_test.shape, "min/max:", X_test.min(), X_test.max(),
      "| lib(mean):", lib_r.mean(), "zero_frac(mean):", z_r.mean())

step  0
step  200
step  400
step  600
step  800
[scLDM] running:
/home/ganchao/miniconda3/envs/vfm311/bin/python experiments/scripts/inference.py ckpt_file=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/experiments/checkpoints/ldm_dg_custom/last.ckpt config_file=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/experiments/checkpoints/ldm_dg_custom/config.yaml inference_path=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/inference dataset_generation_idx=0 datamodule.datamodule.test_batch_size=128 ++model.module.generation_args.timesteps=100 ++model.module.generation_args.guidance_weight={clusters:0.0} ++datamodule.dataset_params.homo_sapiens.metadata_genes=null seed=0 datamodule.dataset=dentate_gyrus datamodule.datamodule.train_adata_path=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/data/train.h5ad datamodule.datamodule.test_adata_path=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/data/test.h5ad datamodule.vocabulary_encoder.adata_path=/home

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/cellarium/ml/utilities/distributed.py:52: UserWarning: Distributed package is available but the default process group has not been initialized. Falling back to ``rank=0`` and ``num_replicas=1``.
  warnings.warn(
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently

[2026-03-09 07:12:39,167][pytorch_lightning.accelerators.cuda][INFO] - LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/cellarium/ml/u00 0.38it/s 0.37it/s 
tilities/distributed.py:52: UserWarning: Distributed package is available but 
the default process group has not been initialized. Falling back to ``rank=0`` 
and ``num_replicas=1``.
  warnings.warn(
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 0:00:13 • 0:00:00 0.38it/s 
INFO     Processing generation output                                           


... storing 'clusters' as categorical


DCM sample cmd (chunk 0, n=58):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 58 --num_steps 100 --temperature 1.0 --seed 0 --val_fraction 0.1 --use_train_split
Using device: cuda
Found best checkpoint: /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/best.pt
Results will be saved to: /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Loaded training configuration from /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/args.json
Model: hidden_dim=256, layers=8, heads=8

Loading data from /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad
Loaded 2344 cells with 13913 genes
Number of gene

...

Generated 58 cells

Saving generated cells to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Generation Statistics
Mean expression correlation: 0.9813
Std expression correlation: 0.7250

Real cells - Mean: 0.22, Std: 2.07
Generated cells - Mean: 0.17, Std: 1.50

Generating visualizations...

Generation complete! Results saved to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results
DCM sample cmd (chunk 9, n=58):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 58 --num_steps 100 --temperature 1.0 --seed 9 --val_fraction 0.1 --use_train_split
Using device: cuda
Found best checkpoint: /home/ganchao/Desktop/count_FM/saved_models

Sampling:   0%|          | 0/100 [00:00<?, ?it/s]/home/ganchao/Desktop/count_FM/aivc-dcm/src/sedd/sampling.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp, dtype=self.amp_dtype):
Sampling: 100%|██████████| 100/100 [01:41<00:00,  1.01s/it]


Generated 58 cells

Saving generated cells to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Generation Statistics
Mean expression correlation: 0.9883
Std expression correlation: 0.8043

Real cells - Mean: 0.21, Std: 1.98
Generated cells - Mean: 0.18, Std: 1.58

Generating visualizations...

Generation complete! Results saved to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results
DCM sample cmd (chunk 10, n=6):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 6 --num_steps 100 --temperature 1.0 --seed 10 --val_fraction 0.1 --use_train_split
Using device: cuda
Found best checkpoint: /home/ganchao/Desktop/count_FM/saved_models

Sampling:   0%|          | 0/100 [00:00<?, ?it/s]/home/ganchao/Desktop/count_FM/aivc-dcm/src/sedd/sampling.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp, dtype=self.amp_dtype):
Sampling: 100%|██████████| 100/100 [00:10<00:00,  9.97it/s]


Generated 6 cells

Saving generated cells to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Generation Statistics
Mean expression correlation: 0.9494
Std expression correlation: 0.7397

Real cells - Mean: 0.17, Std: 1.45
Generated cells - Mean: 0.18, Std: 2.16

Generating visualizations...

Generation complete! Results saved to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results
count-FM: (586, 13913) min/max: 0 184 | lib(mean): 2445.6211604095565 zero_frac(mean): 0.907205896025251
scVI: (586, 13913) min/max: 0 724 | lib(mean): 2417.0546075085326 zero_frac(mean): 0.9090786994460212
scDiffusion: (586, 13913) min/max: 0 2321 | lib(mean): 2913.047781569966 zero_frac(mean): 0.910857795236071
CFGen: (586, 13913) min/max: 0 82831 | lib(mean): 3024.8890784982937 zero_frac(mean): 0.9303078933469789
scLDM: (586, 13913) min/max: 0 923 | lib(mean): 2886.4692832764504 zero_frac(mean): 0.8996639035998694
DCM: (5

# 6. Evaluation

## PCA space

In [41]:
# ---- metrics (rebuild featurizer with principled PCA_DIM) ----

# choose PCA dim by explained variance on REAL test (after gene selection + normalization inside Featurizer)
# goal: keep smallest k such that cumvar >= VAR_TARGET, with bounds [PCA_MIN, PCA_MAX]
VAR_TARGET = 0.90
PCA_MIN = 20
PCA_MAX = 500
# G_KEEP = X_test.shape[1]
G_KEEP = 2000

# IMPORTANT: rebuild feat with pca_dim=None so we can inspect PCA spectrum on the selected genes
feat0 = Featurizer(
    target_sum=TARGET_SUM,
    g_keep=G_KEEP,
    pca_dim=None,
    seed=SEED,
)
feat0.fit(X_test)  # use the evaluation reference distribution
Z0 = feat0.transform(X_test)  # this should be (n, G_KEEP) if your Featurizer is written as intended

# fit PCA spectrum on Z0 to pick k
from sklearn.decomposition import PCA
pca_full = PCA(n_components=min(G_KEEP, Z0.shape[0]-1), random_state=SEED)
pca_full.fit(Z0)

cumvar = np.cumsum(pca_full.explained_variance_ratio_)
k = int(np.searchsorted(cumvar, VAR_TARGET) + 1)
k = max(PCA_MIN, min(PCA_MAX, k))

print(f"[eval] PCA_DIM selected = {k} (cumvar={cumvar[k-1]:.3f}, target={VAR_TARGET}, G_KEEP={G_KEEP})")

# now rebuild the actual featurizer used for metrics with this PCA_DIM
feat = Featurizer(
    target_sum=TARGET_SUM,
    g_keep=G_KEEP,
    pca_dim=k,
    seed=SEED,
)
feat.fit(X_test)

Z_real  = feat.transform(X_test)
Z_cf    = feat.transform(Xhat_countfm)
Z_sv    = feat.transform(Xhat_scvi)
Z_scd   = feat.transform(Xhat_scd)
Z_cg    = feat.transform(Xhat_cfgen)
Z_scldm = feat.transform(Xhat_scldm)
Z_dcm   = feat.transform(Xhat_dcm)

sigma2_fixed = _median_heuristic_sigma2(Z_real, max_points=1000, seed=SEED)
sigma2_fixed = max(sigma2_fixed, 1e-6)

w2_cf    = w2_gaussian(Z_real, Z_cf)
w2_sv    = w2_gaussian(Z_real, Z_sv)
w2_scd   = w2_gaussian(Z_real, Z_scd)
w2_cg    = w2_gaussian(Z_real, Z_cg)
w2_scldm = w2_gaussian(Z_real, Z_scldm)
w2_dcm   = w2_gaussian(Z_real, Z_dcm)

mmd_cf, _    = mmd2_rbf_unbiased(Z_real, Z_cf,    sigma2=sigma2_fixed)
mmd_sv, _    = mmd2_rbf_unbiased(Z_real, Z_sv,    sigma2=sigma2_fixed)
mmd_scd, _   = mmd2_rbf_unbiased(Z_real, Z_scd,   sigma2=sigma2_fixed)
mmd_cg, _    = mmd2_rbf_unbiased(Z_real, Z_cg,    sigma2=sigma2_fixed)
mmd_scldm, _ = mmd2_rbf_unbiased(Z_real, Z_scldm, sigma2=sigma2_fixed)
mmd_dcm, _   = mmd2_rbf_unbiased(Z_real, Z_dcm,   sigma2=sigma2_fixed)

res = pd.DataFrame([
    {"model": "count-FM",    "W2": w2_cf,    "MMD2_RBF": mmd_cf},
    {"model": "scVI",        "W2": w2_sv,    "MMD2_RBF": mmd_sv},
    {"model": "scDiffusion", "W2": w2_scd,   "MMD2_RBF": mmd_scd},
    {"model": "CFGen",       "W2": w2_cg,    "MMD2_RBF": mmd_cg},
    {"model": "scLDM",       "W2": w2_scldm, "MMD2_RBF": mmd_scldm},
    {"model": "DCM",         "W2": w2_dcm,   "MMD2_RBF": mmd_dcm},
]).sort_values("MMD2_RBF")

res

[eval] PCA_DIM selected = 403 (cumvar=0.900, target=0.9, G_KEEP=2000)


,model,W2,MMD2_RBF
0,count-FM,20.497400,0.017625
4,scLDM,20.730997,0.020330
2,scDiffusion,20.604435,0.022320
1,scVI,20.633632,0.022964
5,DCM,23.595008,0.029550
3,CFGen,22.110139,0.035171


In [42]:
# import matplotlib.pyplot as plt

# if Z_real.shape[1] >= 2:
#     plt.figure(figsize=(6,6))
#     plt.scatter(Z_real[:,0],  Z_real[:,1],  s=6, alpha=0.25, label="real test")
#     plt.scatter(Z_cf[:,0],    Z_cf[:,1],    s=6, alpha=0.25, label="count-FM")
#     plt.scatter(Z_sv[:,0],    Z_sv[:,1],    s=6, alpha=0.25, label="scVI")
#     plt.scatter(Z_scd[:,0],   Z_scd[:,1],   s=6, alpha=0.25, label="scDiffusion")
#     plt.scatter(Z_cg[:,0],    Z_cg[:,1],    s=6, alpha=0.25, label="CFGen")
#     plt.scatter(Z_scldm[:,0], Z_scldm[:,1], s=6, alpha=0.25, label="scLDM")
#     plt.scatter(Z_dcm[:,0],   Z_dcm[:,1],   s=6, alpha=0.25, label="DCM")
#     plt.legend()
#     plt.tight_layout()
#     plt.show()

## multiple generations

In [ ]:
# ===== Replication eval (uses current feat) =====
import time, random, numpy as np, torch, pandas as pd

N_REP = 5
GEN_SEEDS = [SEED + i for i in range(N_REP)]
models = ["count-FM", "scVI", "scDiffusion", "CFGen", "scLDM", "DCM"]

# fixed reference features + kernel bandwidth
Z_real = feat.transform(X_test)
sigma2_fixed = _median_heuristic_sigma2(Z_real, max_points=1000, seed=SEED)
sigma2_fixed = max(sigma2_fixed, 1e-6)
print("feat_dim =", Z_real.shape[1], "| sigma2 =", sigma2_fixed)

w2_runs  = {m: [] for m in models}
mmd_runs = {m: [] for m in models}

def _set_all_seeds(s):
    random.seed(int(s))
    np.random.seed(int(s))
    torch.manual_seed(int(s))
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(int(s))

def _scldm_sample_rep(n_samples, n_steps, batch_size, rep_id, unconditional=True):
    # different dataset_generation_idx per rep to avoid identical draws
    n = int(n_samples)
    chunks, got = [], 0
    gen_idx = 1000 * int(rep_id)
    while got < n:
        Xg = scldm_model._infer_one_round(
            gen_idx=gen_idx,
            timesteps=int(n_steps),
            test_batch_size=int(batch_size),
            unconditional=bool(unconditional),
        )
        chunks.append(Xg)
        got += Xg.shape[0]
        gen_idx += 1
    return np.concatenate(chunks, axis=0)[:n]

t0 = time.time()
for r, s in enumerate(GEN_SEEDS):
    print(f"\nrep {r+1}/{N_REP} seed={s}")
    _set_all_seeds(s)

    if hasattr(scdiff_model, "seed"): scdiff_model.seed = int(s)
    if hasattr(dcm_model, "seed"):     dcm_model.seed = int(s)
    if hasattr(scldm_model, "seed"):   scldm_model.seed = int(s)

    # ---- generate ----
    X_cf = countfm_model.sample(n_samples=n_gen, n_step=COUNT_FM_NSTEP, batch_eval=COUNT_FM_BATCH_EVAL,
                                x0_mode="poisson", lam0=COUNT_FM_LAM0)
    X_sv = scvi_model.sample(n_samples=n_gen)
    X_scd = scdiff_model.sample(n_samples=n_gen, batch_size=1000)
    X_cg = cfgen_model.sample(n_samples=n_gen, n_steps=CFGEN_NSTEP, batch_size=CFGEN_BATCH, unconditional=True)
    X_scldm = _scldm_sample_rep(n_gen, SCLDM_NSTEP, SCLDM_BATCH, rep_id=r, unconditional=True)
    X_dcm = dcm_model.sample(n_samples=n_gen, n_steps=DCM_NSTEP, temperature=DCM_TEMP,
                             gen_chunk=max(1, min(32, n_gen)))

    # ---- eval ----
    Z_cf = feat.transform(X_cf)
    Z_sv = feat.transform(X_sv)
    Z_scd = feat.transform(X_scd)
    Z_cg = feat.transform(X_cg)
    Z_scldm = feat.transform(X_scldm)
    Z_dcm = feat.transform(X_dcm)

    w2_runs["count-FM"].append(float(w2_gaussian(Z_real, Z_cf)))
    w2_runs["scVI"].append(float(w2_gaussian(Z_real, Z_sv)))
    w2_runs["scDiffusion"].append(float(w2_gaussian(Z_real, Z_scd)))
    w2_runs["CFGen"].append(float(w2_gaussian(Z_real, Z_cg)))
    w2_runs["scLDM"].append(float(w2_gaussian(Z_real, Z_scldm)))
    w2_runs["DCM"].append(float(w2_gaussian(Z_real, Z_dcm)))

    mmd_runs["count-FM"].append(float(mmd2_rbf_unbiased(Z_real, Z_cf, sigma2=sigma2_fixed)[0]))
    mmd_runs["scVI"].append(float(mmd2_rbf_unbiased(Z_real, Z_sv, sigma2=sigma2_fixed)[0]))
    mmd_runs["scDiffusion"].append(float(mmd2_rbf_unbiased(Z_real, Z_scd, sigma2=sigma2_fixed)[0]))
    mmd_runs["CFGen"].append(float(mmd2_rbf_unbiased(Z_real, Z_cg, sigma2=sigma2_fixed)[0]))
    mmd_runs["scLDM"].append(float(mmd2_rbf_unbiased(Z_real, Z_scldm, sigma2=sigma2_fixed)[0]))
    mmd_runs["DCM"].append(float(mmd2_rbf_unbiased(Z_real, Z_dcm, sigma2=sigma2_fixed)[0]))

print("\nDone in", round(time.time() - t0, 1), "sec")

def _mean_std(xs):
    xs = np.asarray(xs, dtype=float)
    return float(xs.mean()), float(xs.std(ddof=1)) if xs.size > 1 else 0.0

rows = []
for m in models:
    w_mu, w_sd = _mean_std(w2_runs[m])
    k_mu, k_sd = _mean_std(mmd_runs[m])
    rows.append({
        "model": m,
        "W2": f"{w_mu:.3f} ± {w_sd:.3f}",
        "MMD2_RBF": f"{k_mu:.4f} ± {k_sd:.4f}",
        "W2_mean": w_mu, "W2_std": w_sd,
        "MMD2_mean": k_mu, "MMD2_std": k_sd,
    })

res_rep = pd.DataFrame(rows).sort_values("MMD2_mean")
res_rep[["model","W2","MMD2_RBF","W2_mean","W2_std","MMD2_mean","MMD2_std"]]

feat_dim = 403 | sigma2 = 2676.56787109375

rep 1/5 seed=0
step  0
step  200
step  400
step  600
step  800
[scLDM] running:
/home/ganchao/miniconda3/envs/vfm311/bin/python experiments/scripts/inference.py ckpt_file=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/experiments/checkpoints/ldm_dg_custom/last.ckpt config_file=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/experiments/checkpoints/ldm_dg_custom/config.yaml inference_path=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/inference dataset_generation_idx=0 datamodule.datamodule.test_batch_size=128 ++model.module.generation_args.timesteps=100 ++model.module.generation_args.guidance_weight={clusters:0.0} ++datamodule.dataset_params.homo_sapiens.metadata_genes=null seed=0 datamodule.dataset=dentate_gyrus datamodule.datamodule.train_adata_path=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/data/train.h5ad datamodule.datamodule.test_adata_path=/home/ganchao/Desktop/count_FM/saved_models/scldm_dg_v1/da

wandb: WARNING The anonymous setting has no effect and will be removed in a future version.
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/pytorch_lightning/trainer/connectors/data_connector.py:434: The 'predict_dataloader' does not have many workers which may be a bottleneck. Consider increasing the value of the `num_workers` argument` to `num_workers=23` in the `DataLoader` to improve performance.
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/cellarium/ml/utilities/distributed.py:52: UserWarning: Distributed package is available but the default process group has not been initialized. Falling back to ``rank=0`` and ``num_replicas=1``.
  warnings.warn(
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/pytorch_lightning/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently

[2026-03-08 22:42:34,882][pytorch_lightning.accelerators.cuda][INFO] - LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [1]
/home/ganchao/miniconda3/envs/vfm311/lib/python3.11/site-packages/cellarium/ml/u00 0.37it/s 0.36it/s 
tilities/distributed.py:52: UserWarning: Distributed package is available but 
the default process group has not been initialized. Falling back to ``rank=0`` 
and ``num_replicas=1``.
  warnings.warn(
Predicting ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 0:00:14 • 0:00:00 0.37it/s 
INFO     Processing generation output                                           


... storing 'clusters' as categorical


DCM sample cmd (chunk 0, n=32):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 32 --num_steps 100 --temperature 1.0 --seed 0 --val_fraction 0.1 --use_train_split
Using device: cuda
Found best checkpoint: /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/best.pt
Results will be saved to: /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Loaded training configuration from /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/args.json
Model: hidden_dim=256, layers=8, heads=8

Loading data from /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad
Loaded 2344 cells with 13913 genes
Number of gene

...

Generated 32 cells

Saving generated cells to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Generation Statistics
Mean expression correlation: 0.9574
Std expression correlation: 0.7212

Real cells - Mean: 0.21, Std: 1.70
Generated cells - Mean: 0.18, Std: 2.49

Generating visualizations...

Generation complete! Results saved to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results
DCM sample cmd (chunk 11, n=32):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 32 --num_steps 100 --temperature 1.0 --seed 14 --val_fraction 0.1 --use_train_split
Using device: cuda
Found best checkpoint: /home/ganchao/Desktop/count_FM/saved_mode

Sampling:   0%|          | 0/100 [00:00<?, ?it/s]/home/ganchao/Desktop/count_FM/aivc-dcm/src/sedd/sampling.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp, dtype=self.amp_dtype):
Sampling: 100%|██████████| 100/100 [00:53<00:00,  1.87it/s]


Generated 32 cells

Saving generated cells to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Generation Statistics
Mean expression correlation: 0.9854
Std expression correlation: 0.8321

Real cells - Mean: 0.21, Std: 1.72
Generated cells - Mean: 0.18, Std: 1.63

Generating visualizations...

Generation complete! Results saved to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results
DCM sample cmd (chunk 12, n=32):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 32 --num_steps 100 --temperature 1.0 --seed 15 --val_fraction 0.1 --use_train_split
Using device: cuda
Found best checkpoint: /home/ganchao/Desktop/count_FM/saved_mode

Sampling:   0%|          | 0/100 [00:00<?, ?it/s]/home/ganchao/Desktop/count_FM/aivc-dcm/src/sedd/sampling.py:108: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=self.use_amp, dtype=self.amp_dtype):
Sampling: 100%|██████████| 100/100 [00:53<00:00,  1.87it/s]


Generated 32 cells

Saving generated cells to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results

Generation Statistics
Mean expression correlation: 0.9804
Std expression correlation: 0.8037

Real cells - Mean: 0.22, Std: 2.29
Generated cells - Mean: 0.17, Std: 1.51

Generating visualizations...

Generation complete! Results saved to /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints/generation_results
DCM sample cmd (chunk 13, n=32):
/home/ganchao/miniconda3/envs/vfm/bin/python3.9 /home/ganchao/Desktop/count_FM/aivc-dcm/scripts/infernce_generation_maskfix.py --experiment_dir /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/checkpoints --data_path /home/ganchao/Desktop/count_FM/saved_models/dcm_dg_workdir_v1/dcm_train_counts.h5ad --num_generate 32 --num_steps 100 --temperature 1.0 --seed 16 --val_fraction 0.1 --use_train_split


In [ ]:
res_rep

In [ ]:
# # res_rep
# res_rep_old = res_rep.copy()

In [ ]:
# # ===== recompute ONLY count-FM row, keep others from res_rep_old =====
# import numpy as np, time, random, torch, pandas as pd

# # must match the previous replication settings
# N_REP = 5
# GEN_SEEDS = [SEED + i for i in range(N_REP)]

# # these must be the same as used to build res_rep_old
# Z_real = feat.transform(X_test)
# sigma2_fixed = _median_heuristic_sigma2(Z_real, max_points=1000, seed=SEED)
# sigma2_fixed = max(sigma2_fixed, 1e-6)

# def _set_all_seeds(s):
#     random.seed(int(s))
#     np.random.seed(int(s))
#     torch.manual_seed(int(s))
#     if torch.cuda.is_available():
#         torch.cuda.manual_seed_all(int(s))

# w2_list, mmd_list = [], []
# t0 = time.time()
# for r, s in enumerate(GEN_SEEDS):
#     _set_all_seeds(s)

#     X_cf = countfm_model.sample(
#         n_samples=n_gen,
#         n_step=COUNT_FM_NSTEP,
#         batch_eval=COUNT_FM_BATCH_EVAL,
#         x0_mode="poisson",
#         lam0=COUNT_FM_LAM0,
#     )
#     Z_cf = feat.transform(X_cf)

#     w2_list.append(float(w2_gaussian(Z_real, Z_cf)))
#     mmd_list.append(float(mmd2_rbf_unbiased(Z_real, Z_cf, sigma2=sigma2_fixed)[0]))

# print("count-FM recompute time:", round(time.time()-t0, 1), "sec")

# w2_mu = float(np.mean(w2_list))
# w2_sd = float(np.std(w2_list, ddof=1)) if len(w2_list) > 1 else 0.0
# mmd_mu = float(np.mean(mmd_list))
# mmd_sd = float(np.std(mmd_list, ddof=1)) if len(mmd_list) > 1 else 0.0

# new_cf_row = {
#     "model": "count-FM",
#     "W2": f"{w2_mu:.3f} ± {w2_sd:.3f}",
#     "MMD2_RBF": f"{mmd_mu:.4f} ± {mmd_sd:.4f}",
#     "W2_mean": w2_mu, "W2_std": w2_sd,
#     "MMD2_mean": mmd_mu, "MMD2_std": mmd_sd,
# }

# # splice into old table
# res_rep_new = res_rep_old.copy()
# res_rep_new = res_rep_new[res_rep_new["model"] != "count-FM"]
# res_rep_new = pd.concat([res_rep_new, pd.DataFrame([new_cf_row])], ignore_index=True)
# res_rep_new = res_rep_new.sort_values("MMD2_mean")

# res_rep_new[["model","W2","MMD2_RBF","W2_mean","W2_std","MMD2_mean","MMD2_std"]]

In [ ]:
# res_rep_new